# Qwen3-4B Compound Compression Energy Ablation

Reproducible Google Colab experiment for CSE407 Green Computing.

**Primary model:** `Qwen/Qwen3-4B` and `Qwen/Qwen3-4B-AWQ`  
**Hardware:** one NVIDIA T4 16 GB  
**Primary metrics:** GPU energy, power, latency, throughput, VRAM, NIAH retrieval  
**Thinking subset:** exact-answer arithmetic and logic at 8192 input tokens

Run sections in order. The notebook writes every completed run to Google Drive and
can resume after a Colab disconnect. It never silently relabels a fallback as the
requested method.

## Important implementation status

- C0 and C1 use Hugging Face full-precision dynamic KV caches. The T4 has no
  native BF16 support, so C0 uses FP16 and is reported as the full-precision
  baseline. Calling a T4 run "BF16" would be technically misleading.
- C2 first attempts the SHA-pinned community TurboQuant cache. It is accepted only
  after shape, packing, sequence-length, and physical-storage checks. If it fails,
  the actual method is labeled `HF_HQQ_4BIT`, a KIVI-inspired fallback.
- Official KVCache-Factory PyramidKV and standalone SnapKV do **not** support Qwen3.
  They patch Llama/Mistral classes and require incompatible Transformers versions.
  C3/C4 therefore fail closed unless a separately validated Qwen3 adapter is supplied.
  FullKV duplicates are not recorded as eviction results.

This status is intentional research provenance, not a notebook error.

## 1. Install pinned dependencies

Run this cell once, then use **Runtime > Restart session**, and continue at Section 2.
AutoAWQ is deprecated and may downgrade Transformers, so Transformers 5.3.0 is
reinstalled last. The exact TurboQuant community commit is pinned.

In [ ]:
%pip install -q "autoawq==0.2.9" "hqq==0.2.8.post1" "nvidia-ml-py==12.575.51" \
    "pandas==2.3.3" "scipy==1.16.3" "statsmodels==0.14.5" \
    "matplotlib==3.10.7" "seaborn==0.13.2" "accelerate==1.11.0"
%pip install -q --upgrade "transformers==5.3.0"
%pip install -q --no-deps \
    "git+https://github.com/back2matching/turboquant.git@4de496d48bff9a7524016f60396aa374a999c887"
%pip install gptqmodel --no-deps
print("Restart the Colab session before continuing.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Restart the Colab session before continuing.


## 2. Imports, Drive, configuration, and environment manifest

In [ ]:
# Write custom-components module to disk so it can be imported across sessions
custom_py = '"""\nQwen3-4B custom components for CSE407 compound compression ablation.\n\nImplements:\n  - Qwen3PositionAwareCache: KV cache tracking absolute token positions\n  - SnapKV: observation-window importance scoring for KV eviction\n  - diagnostic layer-wise budget allocation\n  - Qwen3Attention monkey-patch: eager attention with position-aware masking\n  - an experimental INT4 cache prototype (not TurboQuant)\n\nAll configs fail closed: if a gate does not pass, the config remains BLOCKED.\n"""\n\nfrom __future__ import annotations\n\nimport gc\nimport math\nfrom dataclasses import dataclass, field\nfrom typing import Any, Callable, Optional\n\nimport torch\nimport torch.nn.functional as F\n\n\n# ---------------------------------------------------------------------------\n# 1. Position-Aware Cache\n# ---------------------------------------------------------------------------\n\nclass Qwen3PositionAwareCache:\n    """Custom KV cache that stores absolute token positions per layer.\n\n    Supports non-contiguous retained positions after eviction.  Implements\n    enough of the Hugging Face ``past_key_values`` protocol to replace\n    ``DynamicCache`` in a patched Qwen3Attention.forward.\n\n    Each layer stores:\n        key_states   [B, num_kv_heads, retained_tokens, head_dim]\n        value_states [B, num_kv_heads, retained_tokens, head_dim]\n        positions    [retained_tokens]   absolute token indices\n    """\n\n    def __init__(self, num_layers: int):\n        self.num_layers = num_layers\n        self._keys: list[Optional[torch.Tensor]] = [None] * num_layers\n        self._values: list[Optional[torch.Tensor]] = [None] * num_layers\n        self._positions: list[Optional[torch.Tensor]] = [None] * num_layers\n        self._logical_lengths: list[int] = [0] * num_layers\n        self._selection_mask: list[Optional[torch.Tensor]] = [None] * num_layers\n\n    # -- Hugging Face past_key_values protocol ---------------------------\n\n    def update(\n        self,\n        key_states: torch.Tensor,\n        value_states: torch.Tensor,\n        layer_idx: int,\n        cache_kwargs: Optional[dict[str, Any]] = None,\n    ) -> tuple[torch.Tensor, torch.Tensor]:\n        """Standard HF entry point; returns the cached K/V for the layer."""\n        # key_states:   [B, num_kv_heads, incoming, head_dim]\n        # value_states: [B, num_kv_heads, incoming, head_dim]\n        cache_kwargs = cache_kwargs or {}\n        cache_position = cache_kwargs.get("cache_position")\n        incoming = key_states.shape[2]\n\n        if cache_position is not None:\n            positions = cache_position.to(key_states.device)\n        else:\n            start = self._logical_lengths[layer_idx]\n            positions = torch.arange(start, start + incoming, device=key_states.device)\n\n        self.set_layer(layer_idx, key_states, value_states, positions, append=True)\n        return self._keys[layer_idx], self._values[layer_idx]\n\n    def get_seq_length(self) -> int:\n        return max(self._logical_lengths) if self._logical_lengths else 0\n\n    def __len__(self) -> int:\n        return self.num_layers\n\n    def __getitem__(self, layer_idx: int):\n        return (\n            self._keys[layer_idx],\n            self._values[layer_idx],\n            self._positions[layer_idx],\n        )\n\n    def __iter__(self):\n        for i in range(self.num_layers):\n            yield i, self[i]\n\n    # -- Extended API ----------------------------------------------------\n\n    def get_layer(self, layer_idx: int):\n        """Return (key_tensor, value_tensor, positions_tensor) or all None."""\n        return self._keys[layer_idx], self._values[layer_idx], self._positions[layer_idx]\n\n    def set_layer(\n        self,\n        layer_idx: int,\n        key_states: torch.Tensor,\n        value_states: torch.Tensor,\n        positions: torch.Tensor,\n        append: bool = False,\n    ) -> None:\n        """Store K/V for a layer, optionally appending to existing cache."""\n        if append and self._keys[layer_idx] is not None:\n            self._keys[layer_idx] = torch.cat(\n                [self._keys[layer_idx], key_states.to(self._keys[layer_idx].device)], dim=2\n            )\n            self._values[layer_idx] = torch.cat(\n                [self._values[layer_idx], value_states.to(self._values[layer_idx].device)], dim=2\n            )\n            self._positions[layer_idx] = torch.cat(\n                [self._positions[layer_idx], positions.to(self._positions[layer_idx].device)]\n            )\n        else:\n            self._keys[layer_idx] = key_states.to(device=key_states.device)\n            self._values[layer_idx] = value_states.to(device=value_states.device)\n            self._positions[layer_idx] = positions.to(device=key_states.device)\n\n        if positions.numel() > 0:\n            self._logical_lengths[layer_idx] = max(\n                self._logical_lengths[layer_idx],\n                int(positions[-1].item()) + 1,\n            )\n\n    def evict_layer(self, layer_idx: int, keep_indices: torch.Tensor) -> None:\n        """Discard all K/V positions NOT in *keep_indices*."""\n        if self._keys[layer_idx] is None:\n            return\n        self._keys[layer_idx] = self._keys[layer_idx][:, :, keep_indices, :]\n        self._values[layer_idx] = self._values[layer_idx][:, :, keep_indices, :]\n        self._positions[layer_idx] = self._positions[layer_idx][keep_indices]\n\n    def logical_length(self, layer_idx: int = 0) -> int:\n        return self._logical_lengths[layer_idx]\n\n    def retained_count(self, layer_idx: int = 0) -> int:\n        k = self._keys[layer_idx]\n        return int(k.shape[2]) if k is not None else 0\n\n    def to(self, device: torch.device) -> "Qwen3PositionAwareCache":\n        for i in range(self.num_layers):\n            if self._keys[i] is not None:\n                self._keys[i] = self._keys[i].to(device)\n            if self._values[i] is not None:\n                self._values[i] = self._values[i].to(device)\n            if self._positions[i] is not None:\n                self._positions[i] = self._positions[i].to(device)\n        return self\n\n\n# ---------------------------------------------------------------------------\n# 2. Utility: GQA head-repeat\n# ---------------------------------------------------------------------------\n\ndef repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:\n    """Repeat KV heads from num_kv_heads to num_heads for GQA."""\n    if n_rep == 1:\n        return hidden_states\n    batch, num_kv_heads, slen, head_dim = hidden_states.shape\n    hidden_states = hidden_states[:, :, None, :, :].expand(\n        batch, num_kv_heads, n_rep, slen, head_dim\n    )\n    return hidden_states.reshape(batch, num_kv_heads * n_rep, slen, head_dim)\n\n\n# ---------------------------------------------------------------------------\n# 3. SnapKV scoring\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass SnapKVConfig:\n    window_size: int = 32          # recent tokens always retained\n    kernel_size: int = 5           # pooling kernel for observation window\n    max_capacity_prompt: int = 1024  # max retained tokens per layer\n\n\ndef snapkv_select(\n    attn_weights: torch.Tensor,\n    query_positions: torch.Tensor,\n    key_positions: torch.Tensor,\n    config: SnapKVConfig,\n    num_key_value_heads: int,\n) -> torch.Tensor:\n    """SnapKV importance-based token selection.\n\n    Args:\n        attn_weights:   [B, num_heads, Q, K]  pre-softmax or post-softmax\n                        attention weights from the *prefill* step.\n        query_positions: [Q] absolute positions of current query tokens.\n        key_positions:   [K] absolute positions of cached+current key tokens.\n        config:          SnapKV parameters.\n        num_key_value_heads: number of KV heads (for GQA aggregation).\n\n    Returns:\n        keep_indices: [K_keep] indices into key_positions of retained tokens.\n    """\n    bsz, num_heads, q_len, k_len = attn_weights.shape\n    n_kv_heads = num_key_value_heads\n    n_q_per_kv = num_heads // n_kv_heads\n    device = attn_weights.device\n\n    # 1. Determine observation window: last *config.window_size* query positions\n    obs_len = min(config.window_size, q_len)\n    obs_queries = attn_weights[:, :, -obs_len:, :]           # [B, H, obs, K]\n\n    # 2. Pool observation weights along the query dimension\n    obs_queries = obs_queries.reshape(bsz, n_kv_heads, n_q_per_kv, obs_len, k_len)\n    obs_queries = obs_queries.mean(dim=2)                     # [B, KV_H, obs, K]\n\n    if config.kernel_size > 1:\n        obs_queries = F.max_pool1d(\n            obs_queries.reshape(bsz * n_kv_heads, obs_len, k_len),\n            kernel_size=config.kernel_size,\n            stride=1,\n            padding=config.kernel_size // 2,\n        ).reshape(bsz, n_kv_heads, obs_len, k_len)\n\n    # 3. Aggregate importance across observation window and KV heads\n    importance = obs_queries.sum(dim=(1, 2))                  # [B, K]\n\n    # 4. Always keep the most recent *config.window_size* tokens\n    window_start = max(0, k_len - config.window_size)\n    window_mask = torch.zeros(k_len, dtype=torch.bool, device=device)\n    window_mask[window_start:] = True\n\n    # 5. Select top-k historical tokens (outside the recent window)\n    historical_mask = ~window_mask\n    n_historical = min(config.max_capacity_prompt - config.window_size, historical_mask.sum().item())\n\n    if n_historical > 0:\n        historical_scores = importance.clone()\n        historical_scores[:, window_mask] = float("-inf")\n        _, top_indices = torch.topk(historical_scores[0], n_historical, dim=-1)\n        keep_mask = window_mask.clone()\n        keep_mask[top_indices] = True\n    else:\n        keep_mask = window_mask\n\n    return torch.where(keep_mask)[0]\n\n\n# ---------------------------------------------------------------------------\n# 4. PyramidKV budget allocation\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass PyramidKVConfig:\n    total_budget: int = 36864      # sum of per-layer token budgets\n    ramp_factor: float = 1.0       # multiplier for the pyramid slope\n\n\ndef pyramidkv_budget(\n    num_layers: int,\n    config: PyramidKVConfig,\n    min_capacity: int = 128,\n) -> list[int]:\n    """Allocate diagnostic per-layer retention budgets using a linear ramp.\n\n    Earlier layers receive fewer tokens (more aggressive compression).\n    Later layers receive more tokens. This is a diagnostic allocation, not\n    a source-faithful PyramidKV reproduction:\n    ``budget_i = base + (total_budget - num_layers * base) * (i / (num_layers-1))^ramp``\n    where ``base`` is computed to meet the total constraint.\n\n    Returns:\n        list of *num_layers* integers summing to *config.total_budget*.\n    """\n    base = min_capacity\n    layers = torch.arange(num_layers, dtype=torch.float32)\n    # Normalised ramp: 0 at layer 0, 1 at last layer\n    ramp = (layers / max(num_layers - 1, 1)) ** config.ramp_factor\n    # Distribute remaining budget proportionally to ramp\n    remaining = max(0, config.total_budget - num_layers * base)\n    weights = ramp / (ramp.sum() + 1e-9)\n    budgets = (base + weights * remaining).round().int().tolist()\n    # Adjust rounding error\n    delta = config.total_budget - sum(budgets)\n    for i in range(abs(delta)):\n        budgets[i % num_layers] += 1 if delta > 0 else -1\n    # Clamp\n    budgets = [max(min_capacity, min(b, config.total_budget)) for b in budgets]\n    return budgets\n\n\ndef pyramidkv_max_capacity_for_layer(\n    layer_idx: int,\n    budgets: list[int],\n    num_key_value_heads: int,\n    head_dim: int,\n) -> int:\n    """PyramidKV per-layer token capacity.\n\n    Uses per-layer budget directly.  At lengths below capacity the layer\n    operates as a FullKV no-op.\n    """\n    return budgets[layer_idx]\n\n\n# ---------------------------------------------------------------------------\n# 5. Qwen3Attention monkey-patch\n# ---------------------------------------------------------------------------\n\ndef create_patched_qwen3_attention(\n    cache: Qwen3PositionAwareCache,\n    eviction_fn: Optional[Callable] = None,\n    force_eager: bool = True,\n) -> Callable:\n    """Produce a replacement ``forward`` method for Qwen3Attention.\n\n    The patched forward:\n      1. Uses native Q/K/V projection, Q/K norm, and RoPE.\n      2. Routes K/V through *cache* instead of DynamicCache.\n      3. Builds a position-aware causal mask.\n      4. Computes attention eagerly (required for score-based eviction).\n      5. After prefill, calls *eviction_fn* to compact the cache.\n      6. On subsequent decode steps, uses only retained positions.\n\n    Args:\n        cache: An existing *Qwen3PositionAwareCache* instance.\n        eviction_fn: ``fn(attn_weights, query_pos, key_pos, layer_idx, cache) -> keep_indices``\n                     Called only during prefill (q_len > 1).  May be None for FullKV.\n        force_eager: If True, always compute attention eagerly (else try SDPA).\n\n    Returns:\n        Patched ``forward(self, hidden_states, position_embeddings, ...)``.\n    """\n    from transformers.models.qwen3.modeling_qwen3 import (\n        Qwen3Attention,\n        apply_rotary_pos_emb,\n    )\n\n    # Capture the original forward so we can route non-patched calls through it.\n    _original_forward = Qwen3Attention.forward\n\n    def patched_forward(\n        self: Qwen3Attention,\n        hidden_states: torch.Tensor,\n        position_embeddings: tuple[torch.Tensor, torch.Tensor],\n        attention_mask: Optional[torch.Tensor] = None,\n        past_key_value: Optional[Any] = None,\n        cache_position: Optional[torch.Tensor] = None,\n        **kwargs,\n    ):\n        # If the caller passes a different cache, delegate to the original.\n        if past_key_value is not cache:\n            return _original_forward(\n                self, hidden_states, position_embeddings,\n                attention_mask, past_key_value, cache_position, **kwargs,\n            )\n\n        bsz, q_len, _ = hidden_states.size()\n\n        # --- Step 1: Native projections ----------------------------------\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        # --- Step 2: Q/K normalization -----------------------------------\n        if self.q_norm is not None:\n            query_states = self.q_norm(query_states)\n        if self.k_norm is not None:\n            key_states = self.k_norm(key_states)\n\n        # --- Step 3: RoPE ------------------------------------------------\n        cos, sin = position_embeddings\n        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)\n\n        # --- Step 4: Determine positions ---------------------------------\n        if cache_position is not None:\n            query_positions = cache_position.to(key_states.device)\n        else:\n            start = cache.logical_length(self.layer_idx)\n            query_positions = torch.arange(start, start + q_len, device=key_states.device)\n\n        # --- Step 5: Retrieve cached K/V ---------------------------------\n        cached_k, cached_v, cached_positions = cache.get_layer(self.layer_idx)\n\n        if cached_k is not None and cached_k.shape[2] > 0:\n            full_k = torch.cat([cached_k.to(key_states.device), key_states], dim=2)\n            full_v = torch.cat([cached_v.to(key_states.device), value_states], dim=2)\n            full_key_positions = torch.cat([\n                cached_positions.to(key_states.device), query_positions\n            ])\n        else:\n            full_k = key_states\n            full_v = value_states\n            full_key_positions = query_positions\n\n        total_k = full_k.shape[2]\n\n        # --- Step 6: GQA repeat ------------------------------------------\n        n_rep = self.num_heads // self.num_key_value_heads\n        key_for_attn = repeat_kv(full_k, n_rep)      # [B, H, total_k, D]\n        value_for_attn = repeat_kv(full_v, n_rep)\n\n        # --- Step 7: Causal mask from actual positions -------------------\n        # [Q, total_k]: True where query_pos >= key_pos\n        causal_mask = (query_positions[:, None] >= full_key_positions[None, :])\n        causal_mask_expanded = causal_mask.unsqueeze(0).unsqueeze(0)  # [1, 1, Q, K]\n\n        # --- Step 8: Eager attention -------------------------------------\n        scale = 1.0 / math.sqrt(self.head_dim)\n        attn_weights = torch.matmul(\n            query_states.to(torch.float32),\n            key_for_attn.transpose(2, 3).to(torch.float32),\n        ) * scale\n\n        attn_weights = attn_weights.masked_fill(\n            ~causal_mask_expanded, float("-inf")\n        )\n\n        # Apply HF attention mask if provided\n        if attention_mask is not None:\n            attn_weights = attn_weights + attention_mask[:, :, :q_len, :total_k]\n\n        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n        attn_weights = F.dropout(attn_weights, p=self.attention_dropout, training=self.training)\n\n        # --- Step 9: Weighted sum ----------------------------------------\n        attn_output = torch.matmul(attn_weights, value_for_attn.to(attn_weights.dtype))\n\n        # --- Step 10: Eviction (prefill only) ----------------------------\n        if eviction_fn is not None and q_len > 1:\n            keep_indices = eviction_fn(\n                attn_weights, query_positions, full_key_positions,\n                self.layer_idx,\n            )\n            full_k = full_k[:, :, keep_indices, :]\n            full_v = full_v[:, :, keep_indices, :]\n            full_key_positions = full_key_positions[keep_indices]\n\n        # --- Step 11: Persist in cache -----------------------------------\n        cache.set_layer(\n            self.layer_idx, full_k, full_v, full_key_positions, append=False,\n        )\n\n        # --- Step 12: Reshape + output projection ------------------------\n        attn_output = attn_output.transpose(1, 2).contiguous().reshape(bsz, q_len, -1)\n        attn_output = self.o_proj(attn_output)\n\n        return attn_output, None, past_key_value\n\n    return patched_forward\n\n\n# ---------------------------------------------------------------------------\n# 6. Sniff original forward (used only to restore)\n# ---------------------------------------------------------------------------\n\n_ORIGINAL_FORWARDS: dict[int, Any] = {}\n\n\ndef sniff_original_forward(model) -> None:\n    """Save a reference to the original Qwen3Attention.forward.\n\n    Call before any monkey-patching.\n    """\n    from transformers.models.qwen3.modeling_qwen3 import Qwen3Attention\n    _ORIGINAL_FORWARDS[id(model)] = Qwen3Attention.forward\n\n\ndef restore_original_forward(model) -> None:\n    """Restore the original Qwen3Attention.forward."""\n    from transformers.models.qwen3.modeling_qwen3 import Qwen3Attention\n    orig = _ORIGINAL_FORWARDS.pop(id(model), None)\n    if orig is not None:\n        for module in model.modules():\n            from transformers.models.qwen3.modeling_qwen3 import Qwen3Attention\n            if isinstance(module, Qwen3Attention):\n                module.forward = orig.__get__(module, Qwen3Attention)\n\n\n# ---------------------------------------------------------------------------\n# 7. FullKV identity gate (no eviction via custom cache)\n# ---------------------------------------------------------------------------\n\ndef build_fullkv_noop_eviction_fn():\n    """Return an eviction function that keeps everything (FullKV identity).\n\n    This lets C1 run through the patched forward without any token drop,\n    enabling an exact equivalence check against the unpatched C1.\n    """\n    def noop_eviction(attn_weights, query_pos, key_pos, layer_idx):\n        return torch.arange(key_pos.shape[0], device=key_pos.device)\n    return noop_eviction\n\n\n# ---------------------------------------------------------------------------\n# 8. SnapKV eviction function factory\n# ---------------------------------------------------------------------------\n\ndef build_snapkv_eviction_fn(\n    snap_config: SnapKVConfig,\n    num_key_value_heads: int,\n) -> Callable:\n    """Return a per-layer eviction function implementing SnapKV selection."""\n    def snapkv_eviction(attn_weights, query_pos, key_pos, layer_idx):\n        return snapkv_select(\n            attn_weights, query_pos, key_pos,\n            snap_config, num_key_value_heads,\n        )\n    return snapkv_eviction\n\n\n# ---------------------------------------------------------------------------\n# 9. PyramidKV eviction function factory\n# ---------------------------------------------------------------------------\n\ndef build_pyramidkv_eviction_fn(\n    snap_config: SnapKVConfig,\n    pyramid_config: PyramidKVConfig,\n    budgets: list[int],\n    num_key_value_heads: int,\n) -> Callable:\n    """Return a per-layer eviction fn: SnapKV scoring + PyramidKV budgets."""\n    def pyramidkv_eviction(attn_weights, query_pos, key_pos, layer_idx):\n        # Per-layer capacity from pyramid budget\n        layer_capacity = budgets[layer_idx]\n        # Temporary SnapKV config with per-layer capacity\n        layer_snap = SnapKVConfig(\n            window_size=snap_config.window_size,\n            kernel_size=snap_config.kernel_size,\n            max_capacity_prompt=layer_capacity,\n        )\n        return snapkv_select(\n            attn_weights, query_pos, key_pos,\n            layer_snap, num_key_value_heads,\n        )\n    return pyramidkv_eviction\n\n\n# ---------------------------------------------------------------------------\n# 10. SnapKV-only eviction (uniform budget — diagnostic/config)\n# ---------------------------------------------------------------------------\n\ndef build_uniform_snapkv_eviction_fn(\n    snap_config: SnapKVConfig,\n    num_key_value_heads: int,\n) -> Callable:\n    """SnapKV with uniform per-layer budget (diagnostic stage).\n\n    This is SnapKV, not PyramidKV.  Use for Section 9 diagnostic gate.\n    """\n    def uniform_snapkv_eviction(attn_weights, query_pos, key_pos, layer_idx):\n        return snapkv_select(\n            attn_weights, query_pos, key_pos,\n            snap_config, num_key_value_heads,\n        )\n    return uniform_snapkv_eviction\n\n\n# ---------------------------------------------------------------------------\n# 11. Cache storage accounting (recursive tensor walker)\n# ---------------------------------------------------------------------------\n\ndef tensor_storage_bytes_custom(root: Any) -> dict[str, Any]:\n    """Recursive tensor storage accounting, categorised by role.\n\n    Returns a dict with per-category byte counts and a total.\n    """\n    seen_objects: set[int] = set()\n    seen_storages: set[tuple[str, int]] = set()\n    categories: dict[str, int] = {\n        "fp16_kv": 0,\n        "quantized_payload": 0,\n        "scale": 0,\n        "residual": 0,\n        "metadata": 0,\n        "other": 0,\n    }\n\n    def visit(value: Any, path: str = "") -> None:\n        oid = id(value)\n        if oid in seen_objects:\n            return\n        seen_objects.add(oid)\n\n        if isinstance(value, torch.Tensor):\n            if value.numel() == 0:\n                return\n            storage = value.untyped_storage()\n            key = (str(value.device), storage.data_ptr())\n            if key in seen_storages:\n                return\n            seen_storages.add(key)\n            nbytes = storage.nbytes()\n            # Categorise by dtype and path hints\n            if value.dtype in (torch.float16, torch.bfloat16):\n                if any(tag in path.lower() for tag in ("key", "value", "k_state", "v_state")):\n                    categories["fp16_kv"] += nbytes\n                else:\n                    categories["other"] += nbytes\n            elif value.dtype == torch.int8 or value.dtype == torch.uint8:\n                if "quant" in path.lower() or "index" in path.lower() or "packed" in path.lower():\n                    categories["quantized_payload"] += nbytes\n                else:\n                    categories["other"] += nbytes\n            elif value.dtype in (torch.float32, torch.float64):\n                categories["scale"] += nbytes\n            else:\n                categories["other"] += nbytes\n            return\n\n        if isinstance(value, dict):\n            for k, v in value.items():\n                visit(v, f"{path}.{k}")\n        elif isinstance(value, (list, tuple, set)):\n            for i, child in enumerate(value):\n                visit(child, f"{path}[{i}]")\n        elif hasattr(value, "__dict__"):\n            for k, v in vars(value).items():\n                if k.startswith("_"):\n                    continue\n                visit(v, f"{path}.{k}")\n\n    visit(root)\n    categories["total"] = sum(categories.values())\n    return categories\n\n\n# ---------------------------------------------------------------------------\n# 12. Validation helpers\n# ---------------------------------------------------------------------------\n\ndef check_token_equivalence(\n    output_unpatched: torch.Tensor,\n    output_patched: torch.Tensor,\n    atol: float = 1e-3,\n) -> bool:\n    """Check that patched and unpatched outputs match within tolerance."""\n    if output_unpatched.shape != output_patched.shape:\n        return False\n    diff = (output_unpatched.float() - output_patched.float()).abs().max().item()\n    return diff <= atol\n\n\ndef check_logit_equivalence(\n    logits_unpatched: torch.Tensor,\n    logits_patched: torch.Tensor,\n    atol: float = 1e-3,\n) -> bool:\n    return check_token_equivalence(logits_unpatched, logits_patched, atol)\n\n\ndef validate_cache_no_fp16_history(cache: Qwen3PositionAwareCache) -> tuple[bool, str]:\n    """Check that no layer stores an FP16 copy of history beyond what is needed."""\n    for layer_idx in range(cache.num_layers):\n        k, v, _ = cache.get_layer(layer_idx)\n        if k is not None and k.dtype in (torch.float16, torch.bfloat16):\n            return False, f"Layer {layer_idx} holds FP16 key cache ({k.shape})"\n        if v is not None and v.dtype in (torch.float16, torch.bfloat16):\n            return False, f"Layer {layer_idx} holds FP16 value cache ({v.shape})"\n    return True, ""\n\n\ndef validate_cache_positions_contiguous(cache: Qwen3PositionAwareCache) -> tuple[bool, str]:\n    """Verify that positions are stored correctly (not necessarily contiguous)."""\n    for layer_idx in range(cache.num_layers):\n        _, _, pos = cache.get_layer(layer_idx)\n        if pos is None:\n            continue\n        if not torch.all(pos.diff() > 0):\n            return False, f"Layer {layer_idx} positions not strictly increasing"\n    return True, ""\n\n\n# ---------------------------------------------------------------------------\n# 13. Experimental compound cache: layerwise eviction + generic INT4\n# ---------------------------------------------------------------------------\n\nclass QuantizedLayerState:\n    """Stores a layer\'s K/V in generic INT4 form; this is not TurboQuant."""\n\n    def __init__(self, quantized_k: torch.Tensor, quantized_v: torch.Tensor,\n                 scales_k: torch.Tensor, scales_v: torch.Tensor,\n                 positions: torch.Tensor, logical_length: int,\n                 original_dtype: torch.dtype = torch.float16):\n        self.quantized_k = quantized_k\n        self.quantized_v = quantized_v\n        self.scales_k = scales_k\n        self.scales_v = scales_v\n        self.positions = positions\n        self.logical_length = logical_length\n        self.original_dtype = original_dtype\n        self._nbits = 4\n\n\nclass CompoundCacheC4(Qwen3PositionAwareCache):\n    """Layerwise-selected K/V compressed with generic per-channel INT4.\n\n    Inherits position tracking from Qwen3PositionAwareCache.  After\n    After layerwise eviction, K/V are quantized using symmetric per-channel\n    INT4. This is an engineering prototype and must not be labelled\n    TurboQuant or used as C4 evidence.\n\n    During attention, dequantized K/V are materialized on-GPU and\n    discarded after the forward pass (they are NOT stored back to the\n    cache).  This proves that the persistent savings are real.\n    """\n\n    def __init__(self, num_layers: int):\n        super().__init__(num_layers)\n        self._quantized_layers: list[Optional[QuantizedLayerState]] = [None] * num_layers\n        self._fp16_kv_freed: bool = False\n        self._nbits = 4\n\n    # --- Quantize / dequantize -------------------------------------------\n\n    def _quantize_tensor(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\n        """Symmetric per-channel INT4 quantization of the last dimension."""\n        # x: [B, H, L, D]\n        amax = x.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8)  # [B, H, L, 1]\n        qmax = 2 ** (self._nbits - 1) - 1                          # 7 for INT4\n        scale = amax / qmax\n        q = (x / scale).round().clamp(-qmax, qmax).to(torch.int8)\n        return q, scale\n\n    def _dequantize_tensor(self, q: torch.Tensor, scale: torch.Tensor,\n                           dtype: torch.dtype) -> torch.Tensor:\n        return (q.to(dtype) * scale.to(dtype))\n\n    # --- Override set_layer to optionally quantize -----------------------\n\n    def set_layer_quantized(\n        self, layer_idx: int,\n        key_states: torch.Tensor, value_states: torch.Tensor,\n        positions: torch.Tensor, append: bool = False,\n    ) -> None:\n        """Store K/V in quantized form, freeing FP16 references."""\n        if append and self._quantized_layers[layer_idx] is not None:\n            # Dequantize existing, concatenate, re-quantize\n            prev = self._quantized_layers[layer_idx]\n            prev_k = self._dequantize_tensor(prev.quantized_k, prev.scales_k, key_states.dtype)\n            prev_v = self._dequantize_tensor(prev.quantized_v, prev.scales_v, value_states.dtype)\n            new_k = torch.cat([prev_k, key_states], dim=2)\n            new_v = torch.cat([prev_v, value_states], dim=2)\n            new_pos = torch.cat([prev.positions, positions])\n        else:\n            new_k = key_states\n            new_v = value_states\n            new_pos = positions\n\n        qk, sk = self._quantize_tensor(new_k)\n        qv, sv = self._quantize_tensor(new_v)\n        dtype = key_states.dtype\n\n        self._quantized_layers[layer_idx] = QuantizedLayerState(\n            qk, qv, sk, sv, new_pos,\n            logical_length=int(new_pos[-1].item()) + 1 if new_pos.numel() > 0 else 0,\n            original_dtype=dtype,\n        )\n        # Set FP16 tensors in parent to dummy empty tensors (freed)\n        self._keys[layer_idx] = None\n        self._values[layer_idx] = None\n        self._positions[layer_idx] = new_pos\n        self._logical_lengths[layer_idx] = self._quantized_layers[layer_idx].logical_length\n\n    def get_layer_quantized(self, layer_idx: int):\n        """Return dequantized (key_states, value_states, positions)."""\n        state = self._quantized_layers[layer_idx]\n        if state is None:\n            return None, None, None\n        k = self._dequantize_tensor(state.quantized_k, state.scales_k, state.original_dtype)\n        v = self._dequantize_tensor(state.quantized_v, state.scales_v, state.original_dtype)\n        return k, v, state.positions\n\n    # Override get_layer to check quantized storage first\n    def get_layer(self, layer_idx: int):\n        if self._quantized_layers[layer_idx] is not None:\n            return self.get_layer_quantized(layer_idx)\n        return super().get_layer(layer_idx)\n\n    def evict_layer(self, layer_idx: int, keep_indices: torch.Tensor) -> None:\n        """Evict after quantizing: operate on quantized tensors directly."""\n        state = self._quantized_layers[layer_idx]\n        if state is not None:\n            state.quantized_k = state.quantized_k[:, :, keep_indices, :]\n            state.quantized_v = state.quantized_v[:, :, keep_indices, :]\n            state.scales_k = state.scales_k[:, :, keep_indices, :]\n            state.scales_v = state.scales_v[:, :, keep_indices, :]\n            state.positions = state.positions[keep_indices]\n            self._positions[layer_idx] = state.positions\n            return\n        super().evict_layer(layer_idx, keep_indices)\n\n\n# ---------------------------------------------------------------------------\n# 14. Experimental post-eviction INT4 attention patch\n# ---------------------------------------------------------------------------\n\ndef create_patched_qwen3_attention_c4(\n    cache: CompoundCacheC4,\n    eviction_fn: Optional[Callable] = None,\n) -> Callable:\n    """Like create_patched_qwen3_attention but quantizes after eviction.\n\n    After the diagnostic selector retains positions, K/V are quantized with\n    per-channel INT4 and the FP16 copies are freed.  During attention,\n    dequantized K/V are materialized, used, and discarded.\n    """\n    from transformers.models.qwen3.modeling_qwen3 import (\n        Qwen3Attention,\n        apply_rotary_pos_emb,\n    )\n    _original_forward = Qwen3Attention.forward\n\n    def patched_forward(\n        self: Qwen3Attention,\n        hidden_states: torch.Tensor,\n        position_embeddings: tuple[torch.Tensor, torch.Tensor],\n        attention_mask: Optional[torch.Tensor] = None,\n        past_key_value: Optional[Any] = None,\n        cache_position: Optional[torch.Tensor] = None,\n        **kwargs,\n    ):\n        if past_key_value is not cache:\n            return _original_forward(\n                self, hidden_states, position_embeddings,\n                attention_mask, past_key_value, cache_position, **kwargs,\n            )\n\n        bsz, q_len, _ = hidden_states.size()\n\n        query_states = self.q_proj(hidden_states)\n        key_states = self.k_proj(hidden_states)\n        value_states = self.v_proj(hidden_states)\n\n        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)\n        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)\n\n        if self.q_norm is not None:\n            query_states = self.q_norm(query_states)\n        if self.k_norm is not None:\n            key_states = self.k_norm(key_states)\n\n        cos, sin = position_embeddings\n        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)\n\n        if cache_position is not None:\n            query_positions = cache_position.to(key_states.device)\n        else:\n            start = cache.logical_length(self.layer_idx)\n            query_positions = torch.arange(start, start + q_len, device=key_states.device)\n\n        cached_k, cached_v, cached_positions = cache.get_layer(self.layer_idx)\n\n        if cached_k is not None and cached_k.shape[2] > 0:\n            full_k = torch.cat([cached_k.to(key_states.device), key_states], dim=2)\n            full_v = torch.cat([cached_v.to(key_states.device), value_states], dim=2)\n            full_key_positions = torch.cat([\n                cached_positions.to(key_states.device), query_positions\n            ])\n        else:\n            full_k = key_states\n            full_v = value_states\n            full_key_positions = query_positions\n\n        total_k = full_k.shape[2]\n\n        n_rep = self.num_heads // self.num_key_value_heads\n        key_for_attn = repeat_kv(full_k, n_rep)\n        value_for_attn = repeat_kv(full_v, n_rep)\n\n        causal_mask = (query_positions[:, None] >= full_key_positions[None, :])\n        causal_mask_expanded = causal_mask.unsqueeze(0).unsqueeze(0)\n\n        scale = 1.0 / math.sqrt(self.head_dim)\n        attn_weights = torch.matmul(\n            query_states.to(torch.float32),\n            key_for_attn.transpose(2, 3).to(torch.float32),\n        ) * scale\n        attn_weights = attn_weights.masked_fill(~causal_mask_expanded, float("-inf"))\n\n        if attention_mask is not None:\n            attn_weights = attn_weights + attention_mask[:, :, :q_len, :total_k]\n\n        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)\n        attn_weights = F.dropout(attn_weights, p=self.attention_dropout, training=self.training)\n\n        attn_output = torch.matmul(attn_weights, value_for_attn.to(attn_weights.dtype))\n\n        # --- Eviction + Quantization (prefill only) ---\n        if eviction_fn is not None and q_len > 1:\n            keep_indices = eviction_fn(\n                attn_weights, query_positions, full_key_positions,\n                self.layer_idx,\n            )\n            full_k = full_k[:, :, keep_indices, :]\n            full_v = full_v[:, :, keep_indices, :]\n            full_key_positions = full_key_positions[keep_indices]\n            # Store quantized, freeing FP16\n            cache.set_layer_quantized(\n                self.layer_idx, full_k, full_v, full_key_positions, append=False,\n            )\n        elif q_len == 1:\n            # Append single new token (quantized if rest is quantized)\n            if cache._quantized_layers[self.layer_idx] is not None:\n                cache.set_layer_quantized(\n                    self.layer_idx, key_states, value_states,\n                    query_positions, append=True,\n                )\n            else:\n                cache.set_layer(\n                    self.layer_idx, key_states, value_states,\n                    query_positions, append=True,\n                )\n        else:\n            cache.set_layer(\n                self.layer_idx, full_k, full_v, full_key_positions, append=False,\n            )\n\n        # --- Attention output ---\n        attn_output = attn_output.transpose(1, 2).contiguous().reshape(bsz, q_len, -1)\n        attn_output = self.o_proj(attn_output)\n\n        # Free dequantized tensors (they were materialized in get_layer_quantized)\n        del full_k, full_v, key_for_attn, value_for_attn, attn_weights\n\n        return attn_output, None, past_key_value\n\n    return patched_forward\n\n\n# ---------------------------------------------------------------------------\n# 15. C4 factory: create compound cache + attention patch\n# ---------------------------------------------------------------------------\n\ndef create_c4_compound_cache_and_patch(\n    model,\n    snap_config: SnapKVConfig,\n    pyramid_config: PyramidKVConfig,\n    budgets: list[int],\n) -> tuple[CompoundCacheC4, Callable]:\n    """Create an experimental layerwise-eviction + INT4 prototype.\n\n    Returns (cache, patched_forward_fn).\n    """\n    cache = CompoundCacheC4(model.config.num_hidden_layers)\n    eviction_fn = build_pyramidkv_eviction_fn(\n        snap_config, pyramid_config, budgets,\n        model.config.num_key_value_heads,\n    )\n    patched_fn = create_patched_qwen3_attention_c4(cache, eviction_fn)\n    return cache, patched_fn\n'
with open("/content/qwen3_custom_components.py", "w", encoding="utf-8") as fh:
    fh.write(custom_py.lstrip())
print("Custom components written to /content/qwen3_custom_components.py")

Custom components written to /content/qwen3_custom_components.py


In [ ]:
from __future__ import annotations

import contextlib
import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import subprocess
import threading
import time
import traceback
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st
import seaborn as sns
import torch
import transformers
from scipy.stats import binomtest, friedmanchisquare, wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from qwen3_custom_components import (
    Qwen3PositionAwareCache,
    CompoundCacheC4,
    create_patched_qwen3_attention,
    create_patched_qwen3_attention_c4,
    create_c4_compound_cache_and_patch,
    sniff_original_forward, restore_original_forward,
    build_snapkv_eviction_fn, build_pyramidkv_eviction_fn,
    build_fullkv_noop_eviction_fn,
    SnapKVConfig, PyramidKVConfig,
    pyramidkv_budget,
    tensor_storage_bytes_custom,
    validate_cache_no_fp16_history,
    validate_cache_positions_contiguous,
    check_token_equivalence, check_logit_equivalence,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache, QuantizedCache

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_ROOT = Path("/content/drive/MyDrive/CSE407_Qwen3_Energy")
except ImportError:
    DEFAULT_ROOT = Path.cwd() / "qwen3_energy_results"

ROOT = Path(os.environ.get("CSE407_RESULTS_DIR", DEFAULT_ROOT))
RAW_DIR = ROOT / "raw"
TRACE_DIR = ROOT / "power_traces"
FIG_DIR = ROOT / "figures"
for directory in (ROOT, RAW_DIR, TRACE_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 407
CONTEXT_LENGTHS = (512, 2048, 8192)
PERFORMANCE_BLOCKS = 10
NIAH_ITEMS_PER_CONDITION = 30
REASONING_ITEMS = 30
DECODE_TOKENS = 128
NIAH_MAX_NEW_TOKENS = 24
THINKING_MAX_NEW_TOKENS = 2048
POLL_INTERVAL_S = 0.1
COOLDOWN_TARGET_C = 72
COOLDOWN_TIMEOUT_S = 120
MODEL_BF16 = "Qwen/Qwen3-4B"
MODEL_AWQ = "Qwen/Qwen3-4B-AWQ"
TURBOQUANT_COMMIT = "4de496d48bff9a7524016f60396aa374a999c887"
RUN_FULL_PERFORMANCE = False
RUN_FULL_NIAH = False
RUN_FULL_THINKING = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULT_COLUMNS = [
    "run_id", "timestamp_utc", "experiment", "block", "item_id", "config",
    "requested_method", "actual_method", "method_validated", "model_id",
    "context_target", "input_tokens", "output_tokens", "thinking",
    "needle_depth", "correct", "response", "prefill_s", "decode_s", "total_s",
    "prefill_tok_s", "decode_tok_s", "avg_power_w", "min_power_w", "max_power_w",
    "gross_energy_j", "idle_power_w", "active_energy_j", "energy_per_output_token_j",
    "torch_loaded_mb", "torch_peak_mb", "nvml_peak_used_mb", "cache_live_mb",
    "power_samples", "trace_file", "valid", "error",
]

def package_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def gpu_manifest() -> dict[str, Any]:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Select a T4 Colab runtime.")
    props = torch.cuda.get_device_properties(0)
    return {
        "timestamp_utc": utc_now(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "autoawq": package_version("autoawq"),
        "turboquant": package_version("turboquant"),
        "hqq": package_version("hqq"),
        "cuda_runtime": torch.version.cuda,
        "gpu_name": props.name,
        "gpu_total_mb": props.total_memory / 2**20,
        "compute_capability": f"{props.major}.{props.minor}",
        "seed": SEED,
        "turboquant_commit": TURBOQUANT_COMMIT,
        "contexts": CONTEXT_LENGTHS,
        "performance_blocks": PERFORMANCE_BLOCKS,
        "niah_items_per_condition": NIAH_ITEMS_PER_CONDITION,
        "baseline_dtype": "float16 (T4 does not support native bfloat16)",
    }

MANIFEST = gpu_manifest()
if "T4" not in MANIFEST["gpu_name"]:
    raise RuntimeError(f"Expected an NVIDIA T4, found {MANIFEST['gpu_name']!r}.")
if transformers.__version__ != "5.3.0":
    raise RuntimeError("This notebook is validated against transformers==5.3.0.")
(ROOT / "environment_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
display(pd.Series(MANIFEST, name="value"))

Mounted at /content/drive


,value
timestamp_utc,2026-07-12T12:11:00.312351+00:00
python,3.12.13
platform,Linux-6.6.122+-x86_64-with-glibc2.35
torch,2.11.0+cu128
transformers,5.3.0
autoawq,0.2.9
turboquant,0.3.1
hqq,0.2.8.post1
cuda_runtime,12.8
gpu_name,Tesla T4


## 3. Atomic persistence and NVML power monitor

Energy is trapezoidally integrated from timestamped samples. CUDA is synchronized
immediately before monitor start and before monitor stop. Raw traces are retained.

In [ ]:
import pynvml

pynvml.nvmlInit()
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def append_csv_atomic(path: Path, row: dict[str, Any], columns=RESULT_COLUMNS) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    lock = path.with_suffix(path.suffix + ".lock")
    while True:
        try:
            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.close(fd)
            break
        except FileExistsError:
            time.sleep(0.05)
    try:
        exists = path.exists() and path.stat().st_size > 0
        with path.open("a", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
            if not exists:
                writer.writeheader()
            writer.writerow({key: row.get(key) for key in columns})
            handle.flush()
            os.fsync(handle.fileno())
    finally:
        lock.unlink(missing_ok=True)

class GPUMonitor:
    def __init__(self, interval_s: float = POLL_INTERVAL_S):
        self.interval_s = interval_s
        self.samples: list[dict[str, float]] = []
        self._running = False
        self._thread: threading.Thread | None = None
        self._t0 = 0.0

    def _sample(self) -> None:
        try:
            memory = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
            util = pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE)
            self.samples.append({
                "t_s": time.perf_counter() - self._t0,
                "power_w": pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0,
                "temperature_c": float(pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)),
                "gpu_util_pct": float(util.gpu),
                "memory_used_mb": memory.used / 2**20,
            })
        except pynvml.NVMLError as exc:
            self.samples.append({"t_s": time.perf_counter() - self._t0, "error": str(exc)})

    def _poll(self) -> None:
        deadline = self._t0
        while self._running:
            self._sample()
            deadline += self.interval_s
            time.sleep(max(0.0, deadline - time.perf_counter()))

    def start(self) -> None:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        self.samples = []
        self._running = True
        self._t0 = time.perf_counter()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self, idle_power_w: float) -> dict[str, float]:
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - self._t0
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)
        self._sample()
        valid = [sample for sample in self.samples if "power_w" in sample]
        if len(valid) < 2:
            raise RuntimeError(f"Only {len(valid)} valid NVML power samples were collected.")
        times = np.asarray([sample["t_s"] for sample in valid], dtype=float)
        powers = np.asarray([sample["power_w"] for sample in valid], dtype=float)
        gross_j = float(np.trapezoid(powers, times))
        return {
            "monitor_elapsed_s": elapsed,
            "avg_power_w": float(gross_j / max(times[-1] - times[0], 1e-9)),
            "min_power_w": float(powers.min()),
            "max_power_w": float(powers.max()),
            "gross_energy_j": gross_j,
            "active_energy_j": max(0.0, gross_j - idle_power_w * (times[-1] - times[0])),
            "nvml_peak_used_mb": max(sample["memory_used_mb"] for sample in valid),
            "power_samples": len(valid),
        }

def measure_idle_power(seconds: float = 30.0) -> float:
    samples = []
    end = time.monotonic() + seconds
    while time.monotonic() < end:
        samples.append(pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0)
        time.sleep(POLL_INTERVAL_S)
    idle = float(np.median(samples))
    (ROOT / "idle_power.json").write_text(
        json.dumps({"timestamp_utc": utc_now(), "seconds": seconds, "median_w": idle, "samples_w": samples}, indent=2),
        encoding="utf-8",
    )
    return idle

IDLE_POWER_W = measure_idle_power(30.0)
print(f"Idle median: {IDLE_POWER_W:.2f} W")

Idle median: 9.66 W


## 4. Exact-token NIAH and reasoning inputs

Inputs are assembled directly as token IDs around a sentinel in Qwen's own chat
template. This avoids the historical notebook's approximate word-to-token conversion.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_BF16, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

SENTINEL = "<CSE407_INSERT_9f31>"
FILLER_TEXT = (
    "Archival note: the committee reviewed routine schedules, equipment inventories, "
    "weather summaries, and ordinary administrative records. No action was required. "
)
NEEDLE = "The secret project code name is SHADOW-8821."

def template_parts(instruction: str, enable_thinking: bool) -> tuple[list[int], list[int]]:
    messages = [
        {"role": "system", "content": "Follow the user's instruction exactly."},
        {"role": "user", "content": f"{instruction}\n\n{SENTINEL}"},
    ]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    if rendered.count(SENTINEL) != 1:
        raise RuntimeError("The chat-template sentinel was not preserved exactly once.")
    left, right = rendered.split(SENTINEL)
    return tokenizer.encode(left, add_special_tokens=False), tokenizer.encode(right, add_special_tokens=False)

def repeated_ids(text: str, count: int) -> list[int]:
    unit = tokenizer.encode(text, add_special_tokens=False)
    if not unit:
        raise ValueError("Filler text tokenized to an empty sequence.")
    return (unit * math.ceil(count / len(unit)))[:count]

def make_niah_input(context_length: int, depth: float, item_id: int) -> dict[str, torch.Tensor]:
    instruction = (
        "Read the document. Complete the final statement using only the code name from the document. "
        "Return the code name and nothing else."
    )
    left, right = template_parts(instruction, enable_thinking=False)
    needle_ids = tokenizer.encode(f"\n{NEEDLE}\n", add_special_tokens=False)
    question_ids = tokenizer.encode("\nThe secret project code name is", add_special_tokens=False)
    available = context_length - len(left) - len(right) - len(needle_ids) - len(question_ids)
    if available < 32:
        raise ValueError(f"Context {context_length} is too short for fixed prompt overhead.")
    rng = random.Random(SEED * 100000 + context_length * 100 + item_id)
    jitter = rng.randint(-max(1, available // 100), max(1, available // 100))
    before = min(available, max(0, round(available * depth) + jitter))
    after = available - before
    ids = left + repeated_ids(FILLER_TEXT, before) + needle_ids + repeated_ids(FILLER_TEXT, after) + question_ids + right
    if len(ids) != context_length:
        raise AssertionError(f"Constructed {len(ids)} tokens, expected {context_length}.")
    return {
        "input_ids": torch.tensor([ids], dtype=torch.long),
        "attention_mask": torch.ones((1, context_length), dtype=torch.long),
    }

def reasoning_dataset(n: int = REASONING_ITEMS) -> list[dict[str, str]]:
    rng = random.Random(SEED)
    rows = []
    for index in range(n):
        a, b, c = rng.randint(12, 90), rng.randint(3, 20), rng.randint(2, 9)
        if index % 3 == 0:
            question, answer = f"What is ({a} + {b}) * {c}?", str((a + b) * c)
        elif index % 3 == 1:
            total = a * c + b
            question = f"A box has {total} cards. After {b} are removed, the rest are split equally among {c} people. How many does each person get?"
            answer = str(a)
        else:
            question = f"Sequence: {a}, {a+b}, {a+2*b}, {a+3*b}. What is the next number?"
            answer = str(a + 4 * b)
        rows.append({"item_id": f"reason_{index:02d}", "question": question, "answer": answer})
    return rows

def make_reasoning_input(item: dict[str, str], context_length: int = 8192) -> dict[str, torch.Tensor]:
    instruction = (
        "Ignore the archival notes, solve the final problem carefully, and put only the final integer in "
        "\\boxed{...}.\n\nArchival notes follow."
    )
    left, right = template_parts(instruction, enable_thinking=True)
    question_ids = tokenizer.encode(f"\n\nProblem: {item['question']}", add_special_tokens=False)
    filler_count = context_length - len(left) - len(right) - len(question_ids)
    if filler_count < 0:
        raise ValueError("Reasoning prompt overhead exceeds target context.")
    ids = left + repeated_ids(FILLER_TEXT, filler_count) + question_ids + right
    assert len(ids) == context_length
    return {
        "input_ids": torch.tensor([ids], dtype=torch.long),
        "attention_mask": torch.ones((1, context_length), dtype=torch.long),
    }

def niah_correct(response: str) -> bool:
    return "shadow-8821" in response.lower()

def extract_boxed_integer(response: str) -> str | None:
    boxed = re.findall(r"\\boxed\s*\{\s*(-?\d+)\s*\}", response)
    if boxed:
        return boxed[-1]
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", response)
    return integers[-1] if integers else None

for length in CONTEXT_LENGTHS:
    assert make_niah_input(length, 0.5, 0)["input_ids"].shape[1] == length
assert make_reasoning_input(reasoning_dataset(1)[0])["input_ids"].shape[1] == 8192
print("Exact-token prompt checks passed.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Exact-token prompt checks passed.


## 5. Cache factories, physical storage accounting, and capability registry

All five planned configurations are defined, but only source-faithful,
validated methods may be collected. The local position-aware cache and
layerwise selector are diagnostics, not a PyramidKV replacement.

In addition to the Hugging‑Face DynamicCache (C0/C1) and TurboQuant fallback
(C2), this section registers:
- ``Qwen3PositionAwareCache`` for C1_patched and diagnostic selector pilots
- SnapKV-style importance scoring with per-layer observation-window aggregation
- a diagnostic layerwise allocation, retained only to test cache/mask invariants

The TurboQuant source audit guards C2. C3/C4 remain blocked until separately
pinned source audits establish method identity; no generic INT4 fallback is
substituted for TurboQuant.

In [ ]:
# --- Physical storage accounting (extended) ---
# The original tensor_storage_bytes covers DynamicCache and QuantizedCache.
# The custom cache exposes tensors via the same Python attribute walk.

# --- SnapKV and PyramidKV configuration ---
SNAPKV_CONFIG = SnapKVConfig(
    window_size=32,
    kernel_size=5,
    max_capacity_prompt=1024,
)

PYRAMIDKV_CONFIG = PyramidKVConfig(
    total_budget=36 * 1024,   # 36,864 layer-token entries
    ramp_factor=1.0,
)
CUSTOM_LAYERWISE_BUDGETS = pyramidkv_budget(36, PYRAMIDKV_CONFIG)
print("Diagnostic layerwise budgets (sum = {}):".format(sum(CUSTOM_LAYERWISE_BUDGETS)))
for i, b in enumerate(CUSTOM_LAYERWISE_BUDGETS):
    if i % 6 == 0:
        print(f"  L{i:02d}-L{min(i+5,35):02d}: {CUSTOM_LAYERWISE_BUDGETS[i:min(i+6,36)]}")

# --- Standard Cache factories ---
def full_cache(model):
    return DynamicCache(config=model.config)

def hqq_cache(model):
    return QuantizedCache(
        backend="hqq",
        config=model.config,
        nbits=4,
        axis_key=1,
        axis_value=1,
        q_group_size=64,
        residual_length=128,
    )

def turboquant_cache(model):
    from turboquant import TurboQuantCache
    return TurboQuantCache(bits=4)

# --- Cache factories for custom cache ---
def qwen3_position_aware_cache_factory(config_code: str, model):
    """Create the appropriate custom cache & eviction for the given config."""
    num_layers = model.config.num_hidden_layers

    if config_code == "C4":
        raise RuntimeError(
            "C4 is blocked: the local INT4 prototype is not source-faithful TurboQuant."
        )

    if config_code in ("C1_patched", "C3_snapkv", "C3"):
        cache = Qwen3PositionAwareCache(num_layers)
    else:
        raise ValueError(f"Unknown custom-cache config: {config_code}")

    if config_code == "C1_patched":
        eviction_fn = build_fullkv_noop_eviction_fn()
    elif config_code == "C3_snapkv":
        eviction_fn = build_snapkv_eviction_fn(
            SNAPKV_CONFIG, model.config.num_key_value_heads,
        )
    elif config_code == "C3":
        eviction_fn = build_pyramidkv_eviction_fn(
            SNAPKV_CONFIG, PYRAMIDKV_CONFIG, CUSTOM_LAYERWISE_BUDGETS,
            model.config.num_key_value_heads,
        )
    return cache, eviction_fn


# --- Capability table ---
@dataclass
class ExperimentConfig:
    code: str
    model_id: str
    requested_method: str
    cache_factory: Callable[[Any], Any] | None
    actual_method: str
    method_validated: bool
    blocked_reason: str = ""
    # C3/C4 extra fields
    custom_cache: bool = False
    eviction_fn_name: str = ""

CONFIGS: dict[str, ExperimentConfig] = {
    "C0": ExperimentConfig("C0", MODEL_BF16, "FullKV", full_cache, "FullKV", True),
    "C1": ExperimentConfig("C1", MODEL_AWQ, "FullKV", full_cache, "FullKV", True),
    "C2": ExperimentConfig(
        "C2", MODEL_AWQ, "TurboQuant-4bit", turboquant_cache,
        "TurboQuant-community-4bit", False,
    ),
    "C3_snapkv": ExperimentConfig(
        "C3_snapkv", MODEL_AWQ, "SnapKV-1024", None, "Custom-SnapKV-style-1024", False,
        "Diagnostic only: requires a pinned, inspected SnapKV reference before collection.",
        custom_cache=True, eviction_fn_name="build_snapkv_eviction_fn",
    ),
    "C3": ExperimentConfig(
        "C3", MODEL_AWQ, "PyramidKV-1024", None, "BLOCKED", False,
        "Blocked: no pinned, inspected PyramidKV allocation source validates this custom ramp.",
        custom_cache=True, eviction_fn_name="build_pyramidkv_eviction_fn",
    ),
    "C4": ExperimentConfig(
        "C4", MODEL_AWQ, "PyramidKV-1024+TurboQuant-4bit", None,
        "BLOCKED", False,
        "Blocked: requires source-faithful, physically validated PyramidKV and TurboQuant adapters.",
        custom_cache=False,
    ),
}

def display_capabilities() -> pd.DataFrame:
    return pd.DataFrame([
        {
            "config": cfg.code,
            "model": cfg.model_id,
            "requested": cfg.requested_method,
            "actual": cfg.actual_method,
            "validated": cfg.method_validated,
            "status": "ready" if cfg.method_validated else cfg.blocked_reason or "pending validation",
        }
        for cfg in CONFIGS.values()
    ])

display(display_capabilities())


Diagnostic layerwise budgets (sum = 36864):
  L00-L05: [128, 179, 230, 282, 333, 384]
  L06-L11: [435, 486, 538, 589, 640, 691]
  L12-L17: [742, 794, 845, 896, 947, 998]
  L18-L23: [1050, 1101, 1152, 1203, 1254, 1306]
  L24-L29: [1357, 1408, 1459, 1510, 1562, 1613]
  L30-L35: [1664, 1715, 1766, 1818, 1869, 1920]


,config,model,requested,actual,validated,status
0,C0,Qwen/Qwen3-4B,FullKV,FullKV,True,ready
1,C1,Qwen/Qwen3-4B-AWQ,FullKV,FullKV,True,ready
2,C2,Qwen/Qwen3-4B-AWQ,TurboQuant-4bit,TurboQuant-community-4bit,False,pending validation
3,C3_snapkv,Qwen/Qwen3-4B-AWQ,SnapKV-1024,Custom-SnapKV-style-1024,False,"Diagnostic only: requires a pinned, inspected ..."
4,C3,Qwen/Qwen3-4B-AWQ,PyramidKV-1024,BLOCKED,False,"Blocked: no pinned, inspected PyramidKV alloca..."
5,C4,Qwen/Qwen3-4B-AWQ,PyramidKV-1024+TurboQuant-4bit,BLOCKED,False,"Blocked: requires source-faithful, physically ..."


## 6. Model loading, attention patching, and cache pilots

The Qwen3Attention monkey-patch applies only to configurations using the
custom position-aware cache.  Other configs (C0, C1 via DynamicCache, C2
via TurboQuant/HQQ) use the original Hugging-Face forward.

The patch saves the original forward before any modification, so the
unpatched path is always accessible by passing ``past_key_value != cache``
to the forward method.

In [ ]:
# --- Force clear CUDA memory from previous failed runs ---
import gc
import sys
import torch

# Delete existing model variables from the global scope
for var_name in ["bf16_model", "awq_model", "model", "cache", "output", "noop_reference"]:
    if var_name in globals():
        try:
            del globals()[var_name]
        except KeyError:
            pass

# Clear IPython history and cached output variables to release references
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('bf16_model', None)
        ipython.user_ns.pop('awq_model', None)
        ipython.user_ns.pop('model', None)
        ipython.user_ns.pop('noop_reference', None)
        ipython.user_ns.update({
            '_': None, '__': None, '___': None,
            '_i': '', '_ii': '', '_iii': ''
        })
        if hasattr(ipython, 'history_manager'):
            ipython.history_manager.output_clean()
except Exception:
    pass

# Aggressive PyTorch tensor sweep: find all CUDA tensors and reset their data to CPU
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
            if obj.is_cuda:
                obj.data = torch.empty(0, device="cpu")
    except Exception:
        pass

# Run garbage collection multiple times to resolve cyclic dependencies
for _ in range(5):
    gc.collect()

# Empty PyTorch cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


# --- Self-healing dependency installer for gptqmodel ---
attempted_installs = set()
while True:
    try:
        import gptqmodel
        break
    except ModuleNotFoundError as e:
        missing_module = e.name
        if not missing_module:
            import re
            match = re.search(r"No module named '([^']+)'", str(e))
            missing_module = match.group(1) if match else None

        if not missing_module:
            raise e

        # Map module names to PyPI package names if they differ
        package_map = {
            "pcre": "pypcre",
            "logbar": "logbar"
        }
        package_name = package_map.get(missing_module, missing_module)

        if package_name in attempted_installs:
            print(f"Failed to resolve {missing_module!r} by installing {package_name}. Aborting.")
            raise e

        attempted_installs.add(package_name)
        print(f"gptqmodel dependency {missing_module!r} is missing. Installing {package_name}...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


# --- Patch Qwen3PositionAwareCache to support transformers 5.3.0 Cache protocol ---
from qwen3_custom_components import Qwen3PositionAwareCache

# Explicitly implement Cache protocol methods required for mask construction and usability
def get_mask_sizes(self, cache_position, layer_idx=None):
    if hasattr(cache_position, "shape"):
        query_length = cache_position.shape[-1]
    elif isinstance(cache_position, int):
        query_length = 1
    else:
        query_length = 0

    if layer_idx is not None and layer_idx < len(self._logical_lengths):
        seq_len = self._logical_lengths[layer_idx]
    else:
        seq_len = max(self._logical_lengths) if self._logical_lengths else 0

    return seq_len + query_length, 0

def get_seq_length(self, layer_idx=None):
    if layer_idx is not None and layer_idx < len(self._logical_lengths):
        return self._logical_lengths[layer_idx]
    return max(self._logical_lengths) if self._logical_lengths else 0

def get_usable_length(self, new_seq_length, layer_idx=None):
    return self.get_seq_length(layer_idx)

def get_max_length(self):
    return None

Qwen3PositionAwareCache.get_mask_sizes = get_mask_sizes
Qwen3PositionAwareCache.get_seq_length = get_seq_length
Qwen3PositionAwareCache.get_usable_length = get_usable_length
Qwen3PositionAwareCache.get_max_length = get_max_length


# --- Monkey-patch caching allocator warmup ---
import transformers.modeling_utils
transformers.modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None


# --- Helper functions ---
def tensor_storage_bytes(root: Any) -> int:
    """Count unique tensor storages reachable through Python attributes/containers."""
    seen_objects: set[int] = set()
    seen_storages: set[tuple[str, int]] = set()
    total = 0

    def visit(value: Any) -> None:
        nonlocal total
        object_id = id(value)
        if object_id in seen_objects:
            return
        seen_objects.add(object_id)
        if isinstance(value, torch.Tensor):
            if value.numel() == 0:
                return
            storage = value.untyped_storage()
            key = (str(value.device), storage.data_ptr())
            if key not in seen_storages:
                seen_storages.add(key)
                total += storage.nbytes()
            return
        if isinstance(value, dict):
            for child in value.values():
                visit(child)
        elif isinstance(value, (list, tuple, set)):
            for child in value:
                visit(child)
        elif hasattr(value, "__dict__"):
            for child in vars(value).values():
                visit(child)

    visit(root)
    return total

def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_model(model_id: str):
    clear_cuda()
    kwargs = {
        "device_map": {"": 0},
        "attn_implementation": "sdpa",
        "low_cpu_mem_usage": True,
        "trust_remote_code": False,
    }
    if model_id == MODEL_BF16:
        kwargs["dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    if model.config.model_type != "qwen3":
        raise RuntimeError(f"Expected model_type='qwen3', got {model.config.model_type!r}.")
    if model.config.num_hidden_layers != 36 or model.config.num_key_value_heads != 8 or model.config.head_dim != 128:
        raise RuntimeError("Qwen3-4B architecture does not match the preregistered 36-layer, 8-KV-head, 128-dim design.")
    return model

def make_robust_attention_forward(original_forward, patched_forward_fn, cache):
    def robust_forward(self, *args, **kwargs):
        # Set missing attributes dynamically on self to prevent AttributeErrors in custom forward
        if not hasattr(self, "num_heads"):
            self.num_heads = self.config.num_attention_heads
        if not hasattr(self, "num_key_value_heads"):
            self.num_key_value_heads = self.config.num_key_value_heads

        pkv = kwargs.get("past_key_value", None)
        if pkv is None:
            pkv = kwargs.get("past_key_values", None)
        if pkv is None and len(args) > 3:
            pkv = args[3]

        if pkv is not cache:
            # Delegate to original forward with exact arguments
            return original_forward(self, *args, **kwargs)

        # Clean kwargs to avoid duplicate parameter bindings
        clean_kwargs = kwargs.copy()
        if "past_key_values" in clean_kwargs:
            pkv_val = clean_kwargs.pop("past_key_values")
            if "past_key_value" not in clean_kwargs:
                clean_kwargs["past_key_value"] = pkv_val
        res = patched_forward_fn(self, *args, **clean_kwargs)
        if isinstance(res, tuple) and len(res) == 3:
            return res[0], res[1]
        return res

    return robust_forward

def patch_model_for_custom_cache(model, config_code: str):
    """Attach a position-aware cache and eviction to the model.

    Returns (cache, eviction_fn, patched_model).

    C4 is intentionally blocked unless a separately validated, source-faithful
    compound adapter is supplied.
    """
    result = qwen3_position_aware_cache_factory(config_code, model)
    cache, eviction_fn = result
    patched_fn = create_patched_qwen3_attention(cache, eviction_fn, force_eager=True)
    from transformers.models.qwen3.modeling_qwen3 import Qwen3Attention

    # Save the original unpatched forward from class
    orig_forward = Qwen3Attention.forward

    for module in model.modules():
        if isinstance(module, Qwen3Attention):
            # Wrap the patched function with our robust interface
            robust_fn = make_robust_attention_forward(orig_forward, patched_fn, cache)
            module.forward = robust_fn.__get__(module, Qwen3Attention)
    return cache, eviction_fn

def validate_full_cache(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 999)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    cache = full_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    assert output.past_key_values is cache
    assert int(cache.get_seq_length()) == 512
    live = tensor_storage_bytes(cache)
    assert live > 0
    return {"actual_method": "FullKV", "cache_live_bytes": live, "seq_length": 512}

def validate_custom_cache_noop(model, cache, reference_logits: torch.Tensor) -> dict[str, Any]:
    """Gate 1: custom-cache logits and greedy token match unpatched C1."""
    inputs = make_niah_input(512, 0.5, 997)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    max_logit_error = (output.logits.float() - reference_logits.float()).abs().max().item()
    if max_logit_error > 1e-1:
        raise AssertionError(f"Custom-cache logits diverged from eager C1 (max error {max_logit_error:.6f}).")
    if not torch.equal(output.logits[:, -1, :].argmax(dim=-1), reference_logits[:, -1, :].argmax(dim=-1)):
        raise AssertionError("Custom-cache greedy continuation differs from eager C1.")
    assert output.past_key_values is cache
    assert cache.get_seq_length() == 512
    for layer_idx in range(cache.num_layers):
        k, v, pos = cache.get_layer(layer_idx)
        assert k is not None and v is not None, f"Layer {layer_idx} missing cache"
        assert k.shape[2] == 512, f"Layer {layer_idx}: expected 512 tokens, got {k.shape[2]}"
        assert pos.shape[0] == 512 and pos[-1].item() == 511
    live = tensor_storage_bytes(cache)
    return {
        "actual_method": "FullKV (custom cache)", "cache_live_bytes": live,
        "seq_length": 512, "max_logit_error": max_logit_error,
    }

def validate_turboquant_or_fallback(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 998)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    errors = []
    try:
        cache = turboquant_cache(model)
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
        assert output.past_key_values is cache
        assert int(cache.get_seq_length()) == 512
        assert len(cache.layers) == model.config.num_hidden_layers
        layer = cache.layers[0]
        assert layer._key_indices.dtype == torch.uint8
        assert layer._key_indices.shape[-1] == model.config.head_dim // 2
        assert layer._value_indices.shape[-1] == model.config.head_dim // 2
        live = tensor_storage_bytes(cache)
        reported = cache.memory_usage_bytes()
        if live <= 0 or reported["compressed_bytes"] <= 0:
            raise AssertionError("TurboQuant did not expose non-empty compressed storage.")
        fp16_cache_bytes = (
            model.config.num_hidden_layers
            * model.config.num_key_value_heads
            * 512
            * model.config.head_dim
            * 2
            * 2
        )
        if live >= fp16_cache_bytes:
            raise AssertionError(
                f"TurboQuant compressed indices exist, but reachable live cache storage "
                f"({live / 2**20:.1f} MiB) is not below the FP16 equivalent "
                f"({fp16_cache_bytes / 2**20:.1f} MiB)."
            )
        return {
            "actual_method": "TurboQuant-community-4bit",
            "cache_factory": turboquant_cache,
            "cache_live_bytes": live,
            "reported": reported,
            "errors": errors,
        }
    except Exception as exc:
        errors.append(f"TurboQuant rejected: {type(exc).__name__}: {exc}")
        clear_cuda()

    cache = hqq_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    assert output.past_key_values is cache
    assert int(cache.get_seq_length()) == 512
    live = tensor_storage_bytes(cache)
    if live <= 0:
        raise AssertionError("HQQ fallback did not expose cache storage.")
    fp16_cache_bytes = (
        model.config.num_hidden_layers
        * model.config.num_key_value_heads
        * 512
        * model.config.head_dim
        * 2
        * 2
    )
    if live >= fp16_cache_bytes:
        raise AssertionError("HQQ fallback did not reduce reachable storage below the FP16 cache.")
    return {
        "actual_method": "HF_HQQ_4BIT_KIVI_INSPIRED",
        "cache_factory": hqq_cache,
        "cache_live_bytes": live,
        "errors": errors,
    }

# --- Run pilots ---
# C0 / C1 with standard DynamicCache
pilot_rows = []
bf16_model = load_model(MODEL_BF16)
pilot_rows.append({"config": "C0", **validate_full_cache(bf16_model)})
del bf16_model
clear_cuda()

awq_model = load_model(MODEL_AWQ)
pilot_rows.append({"config": "C1", **validate_full_cache(awq_model)})

# C1_patched: custom cache with no-op eviction (equivalence gate)
noop_inputs = make_niah_input(512, 0.5, 997)
noop_inputs = {key: value.to(awq_model.device) for key, value in noop_inputs.items()}
with torch.inference_mode():
    noop_reference = awq_model(
        **noop_inputs, past_key_values=full_cache(awq_model), use_cache=True, logits_to_keep=1
    ).logits.detach()
sniff_original_forward(awq_model)
cache_c1p, _ = patch_model_for_custom_cache(awq_model, "C1_patched")
pilot_rows.append({"config": "C1_patched", **validate_custom_cache_noop(awq_model, cache_c1p, noop_reference)})
# Restore for next pilot
restore_original_forward(awq_model)

# C2 accepts only the requested, physically validated TurboQuant method.
# A working HQQ path is preserved as a distinct exploratory configuration.
c2_pilot = validate_turboquant_or_fallback(awq_model)
pilot_rows.append({key: value for key, value in {"config": "C2", **c2_pilot}.items() if key != "cache_factory"})
if c2_pilot["actual_method"] == "TurboQuant-community-4bit":
    CONFIGS["C2"] = ExperimentConfig(
        "C2", MODEL_AWQ, "TurboQuant-4bit", turboquant_cache,
        "TurboQuant-community-4bit", True,
    )
else:
    CONFIGS["C2"] = ExperimentConfig(
        "C2", MODEL_AWQ, "TurboQuant-4bit", None, "BLOCKED", False,
        "; ".join(c2_pilot.get("errors", [])),
    )
    CONFIGS["C2_HQQ"] = ExperimentConfig(
        "C2_HQQ", MODEL_AWQ, "HQQ-4bit exploratory", hqq_cache,
        "HF_HQQ_4BIT_KIVI_INSPIRED", True,
        "Exploratory configuration; not TurboQuant.",
    )

# C3_snapkv: SnapKV with uniform budget at 512 (no-op)
sniff_original_forward(awq_model)
cache_c3s, _ = patch_model_for_custom_cache(awq_model, "C3_snapkv")
pilot_rows.append({"config": "C3_snapkv", **validate_custom_cache_noop(awq_model, cache_c3s, noop_reference)})
restore_original_forward(awq_model)

del awq_model
clear_cuda()

(ROOT / "cache_pilots.json").write_text(json.dumps(pilot_rows, indent=2, default=str), encoding="utf-8")
display(pd.DataFrame(pilot_rows))
display(display_capabilities())

/tmp/ipykernel_1119/1825201331.py:35: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

INFO  Kernel: Auto-selection: adding candidate `AwqGEMMTritonLinear`           


INFO  Kernel: selected -> `AwqGEMMTritonLinear`.                               


Loading weights:   0%|          | 0/902 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6132 objects in 0.343s                            


,config,actual_method,cache_live_bytes,seq_length,max_logit_error,errors
0,C0,FullKV,75497472,512.0,NaN,NaN
1,C1,FullKV,75497472,512.0,NaN,NaN
2,C1_patched,FullKV (custom cache),75644928,512.0,0.085938,NaN
3,C2,HF_HQQ_4BIT_KIVI_INSPIRED,21233664,NaN,NaN,[TurboQuant rejected: AssertionError: TurboQua...
4,C3_snapkv,FullKV (custom cache),75644928,512.0,0.085938,NaN


,config,model,requested,actual,validated,status
0,C0,Qwen/Qwen3-4B,FullKV,FullKV,True,ready
1,C1,Qwen/Qwen3-4B-AWQ,FullKV,FullKV,True,ready
2,C2,Qwen/Qwen3-4B-AWQ,TurboQuant-4bit,BLOCKED,False,TurboQuant rejected: AssertionError: TurboQuan...
3,C3_snapkv,Qwen/Qwen3-4B-AWQ,SnapKV-1024,Custom-SnapKV-style-1024,False,"Diagnostic only: requires a pinned, inspected ..."
4,C3,Qwen/Qwen3-4B-AWQ,PyramidKV-1024,BLOCKED,False,"Blocked: no pinned, inspected PyramidKV alloca..."
5,C4,Qwen/Qwen3-4B-AWQ,PyramidKV-1024+TurboQuant-4bit,BLOCKED,False,"Blocked: requires source-faithful, physically ..."
6,C2_HQQ,Qwen/Qwen3-4B-AWQ,HQQ-4bit exploratory,HF_HQQ_4BIT_KIVI_INSPIRED,True,ready


## 6b. SnapKV validation gate at 2048 and 8192

C3_snapkv must correctly evict tokens above its 1024 capacity at long
contexts while preserving NIAH retrieval.  This gate runs a single NIAH
item at each context length.  C3_snapkv is accepted only when both the
structural gate (position tracking, budget) and the NIAH gate pass.

In [ ]:
# --- Force clear CUDA memory from previous failed runs ---
import gc
import sys
import torch
from typing import Any, Callable, Optional

# Delete existing model variables from the global scope
for var_name in ["bf16_model", "awq_model", "model", "cache", "output", "noop_reference"]:
    if var_name in globals():
        try:
            del globals()[var_name]
        except KeyError:
            pass

# Clear IPython history and cached output variables to release references
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.user_ns.pop('bf16_model', None)
        ipython.user_ns.pop('awq_model', None)
        ipython.user_ns.pop('model', None)
        ipython.user_ns.pop('noop_reference', None)
        ipython.user_ns.update({
            '_': None, '__': None, '___': None,
            '_i': '', '_ii': '', '_iii': ''
        })
        if hasattr(ipython, 'history_manager'):
            ipython.history_manager.output_clean()
except Exception:
    pass

# Aggressive PyTorch tensor sweep: find all CUDA tensors and reset their data to CPU
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
            if obj.is_cuda:
                obj.data = torch.empty(0, device="cpu")
    except Exception:
        pass

# Run garbage collection multiple times to resolve cyclic dependencies
for _ in range(5):
    gc.collect()

# Empty PyTorch cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


# --- Self-healing dependency installer for gptqmodel ---
attempted_installs = set()
while True:
    try:
        import gptqmodel
        break
    except ModuleNotFoundError as e:
        missing_module = e.name
        if not missing_module:
            import re
            match = re.search(r"No module named '([^']+)'", str(e))
            missing_module = match.group(1) if match else None

        if not missing_module:
            raise e

        # Map module names to PyPI package names if they differ
        package_map = {
            "pcre": "pypcre",
            "logbar": "logbar"
        }
        package_name = package_map.get(missing_module, missing_module)

        if package_name in attempted_installs:
            print(f"Failed to resolve {missing_module!r} by installing {package_name}. Aborting.")
            raise e

        attempted_installs.add(package_name)
        print(f"gptqmodel dependency {missing_module!r} is missing. Installing {package_name}...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


# --- Patch Qwen3PositionAwareCache to support transformers 5.3.0 Cache protocol ---
from qwen3_custom_components import Qwen3PositionAwareCache

# Explicitly implement Cache protocol methods required for mask construction and usability
def get_mask_sizes(self, cache_position, layer_idx=None):
    if hasattr(cache_position, "shape"):
        query_length = cache_position.shape[-1]
    elif isinstance(cache_position, int):
        query_length = 1
    else:
        query_length = 0

    if layer_idx is not None and layer_idx < len(self._logical_lengths):
        seq_len = self._logical_lengths[layer_idx]
    else:
        seq_len = max(self._logical_lengths) if self._logical_lengths else 0

    return seq_len + query_length, 0

def get_seq_length(self, layer_idx=None):
    if layer_idx is not None and layer_idx < len(self._logical_lengths):
        return self._logical_lengths[layer_idx]
    return max(self._logical_lengths) if self._logical_lengths else 0

def get_usable_length(self, new_seq_length, layer_idx=None):
    return self.get_seq_length(layer_idx)

def get_max_length(self):
    return None

Qwen3PositionAwareCache.get_mask_sizes = get_mask_sizes
Qwen3PositionAwareCache.get_seq_length = get_seq_length
Qwen3PositionAwareCache.get_usable_length = get_usable_length
Qwen3PositionAwareCache.get_max_length = get_max_length


# --- Monkey-patch caching allocator warmup ---
import transformers.modeling_utils
transformers.modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None


# --- Helper functions ---
def tensor_storage_bytes(root: Any) -> int:
    """Count unique tensor storages reachable through Python attributes/containers."""
    seen_objects: set[int] = set()
    seen_storages: set[tuple[str, int]] = set()
    total = 0

    def visit(value: Any) -> None:
        nonlocal total
        object_id = id(value)
        if object_id in seen_objects:
            return
        seen_objects.add(object_id)
        if isinstance(value, torch.Tensor):
            if value.numel() == 0:
                return
            storage = value.untyped_storage()
            key = (str(value.device), storage.data_ptr())
            if key not in seen_storages:
                seen_storages.add(key)
                total += storage.nbytes()
            return
        if isinstance(value, dict):
            for child in value.values():
                visit(child)
        elif isinstance(value, (list, tuple, set)):
            for child in value:
                visit(child)
        elif hasattr(value, "__dict__"):
            for child in vars(value).values():
                visit(child)

    visit(root)
    return total

def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_model(model_id: str):
    clear_cuda()
    kwargs = {
        "device_map": {"": 0},
        "attn_implementation": "eager",
        "low_cpu_mem_usage": True,
        "trust_remote_code": False,
    }
    if model_id == MODEL_BF16:
        kwargs["dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    if model.config.model_type != "qwen3":
        raise RuntimeError(f"Expected model_type='qwen3', got {model.config.model_type!r}.")
    if model.config.num_hidden_layers != 36 or model.config.num_key_value_heads != 8 or model.config.head_dim != 128:
        raise RuntimeError("Qwen3-4B architecture does not match the preregistered 36-layer, 8-KV-head, 128-dim design.")
    return model

def make_robust_attention_forward(original_forward, patched_forward_fn, cache):
    def robust_forward(self, *args, **kwargs):
        # Set missing attributes dynamically on self to prevent AttributeErrors in custom forward
        if not hasattr(self, "num_heads"):
            self.num_heads = self.config.num_attention_heads
        if not hasattr(self, "num_key_value_heads"):
            self.num_key_value_heads = self.config.num_key_value_heads

        pkv = kwargs.get("past_key_value", None)
        if pkv is None:
            pkv = kwargs.get("past_key_values", None)
        if pkv is None and len(args) > 3:
            pkv = args[3]

        if pkv is not cache:
            # Delegate to original forward with exact arguments
            return original_forward(self, *args, **kwargs)

        # Clean kwargs to avoid duplicate parameter bindings
        clean_kwargs = kwargs.copy()
        if "past_key_values" in clean_kwargs:
            pkv_val = clean_kwargs.pop("past_key_values")
            if "past_key_value" not in clean_kwargs:
                clean_kwargs["past_key_value"] = pkv_val

        res = patched_forward_fn(self, *args, **clean_kwargs)
        if isinstance(res, tuple) and len(res) == 3:
            return res[0], res[1]
        return res

    return robust_forward

# --- Dynamic monkey-patch for memory-efficient attention ---
from qwen3_custom_components import repeat_kv

def create_patched_qwen3_attention_optimized(
    cache: Qwen3PositionAwareCache,
    eviction_fn: Optional[Callable] = None,
    force_eager: bool = True,
) -> Callable:
    from transformers.models.qwen3.modeling_qwen3 import (
        Qwen3Attention,
        apply_rotary_pos_emb,
    )
    _original_forward = Qwen3Attention.forward

    def patched_forward(
        self: Qwen3Attention,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[Any] = None,
        cache_position: Optional[torch.Tensor] = None,
        **kwargs,
    ):
        if past_key_value is not cache:
            return _original_forward(
                self, hidden_states, position_embeddings,
                attention_mask, past_key_value, cache_position, **kwargs,
            )

        bsz, q_len, _ = hidden_states.size()

        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        if self.q_norm is not None:
            query_states = self.q_norm(query_states)
        if self.k_norm is not None:
            key_states = self.k_norm(key_states)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if cache_position is not None:
            query_positions = cache_position.to(key_states.device)
        else:
            start = cache.logical_length(self.layer_idx)
            query_positions = torch.arange(start, start + q_len, device=key_states.device)

        cached_k, cached_v, cached_positions = cache.get_layer(self.layer_idx)

        if cached_k is not None and cached_k.shape[2] > 0:
            full_k = torch.cat([cached_k.to(key_states.device), key_states], dim=2)
            full_v = torch.cat([cached_v.to(key_states.device), value_states], dim=2)
            full_key_positions = torch.cat([
                cached_positions.to(key_states.device), query_positions
            ])
        else:
            full_k = key_states
            full_v = value_states
            full_key_positions = query_positions

        total_k = full_k.shape[2]

        n_rep = self.num_heads // self.num_key_value_heads
        key_for_attn = repeat_kv(full_k, n_rep)
        value_for_attn = repeat_kv(full_v, n_rep)

        causal_mask = (query_positions[:, None] >= full_key_positions[None, :])

        is_causal = (attention_mask is None) and (q_len > 1)
        attn_output = torch.nn.functional.scaled_dot_product_attention(
            query_states,
            key_for_attn,
            value_for_attn,
            attn_mask=attention_mask,
            dropout_p=self.attention_dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        if eviction_fn is not None and q_len > 1:
            obs_len = min(32, q_len)
            query_states_slice = query_states[:, :, -obs_len:, :]
            scale = 1.0 / math.sqrt(self.head_dim)
            attn_weights_slice = torch.matmul(
                query_states_slice.to(torch.float32),
                key_for_attn.transpose(2, 3).to(torch.float32),
            ) * scale

            causal_mask_slice = causal_mask[-obs_len:, :]
            causal_mask_expanded_slice = causal_mask_slice.unsqueeze(0).unsqueeze(0)
            attn_weights_slice = attn_weights_slice.masked_fill(
                ~causal_mask_expanded_slice, float("-inf")
            )
            if attention_mask is not None:
                attn_weights_slice = attn_weights_slice + attention_mask[:, :, -obs_len:, :total_k]

            attn_weights_slice = torch.nn.functional.softmax(attn_weights_slice, dim=-1, dtype=torch.float32).to(query_states.dtype)

            keep_indices = eviction_fn(
                attn_weights_slice, query_positions[-obs_len:], full_key_positions,
                self.layer_idx,
            )
            full_k = full_k[:, :, keep_indices, :]
            full_v = full_v[:, :, keep_indices, :]
            full_key_positions = full_key_positions[keep_indices]

        cache.set_layer(
            self.layer_idx, full_k, full_v, full_key_positions, append=False,
        )

        attn_output = attn_output.transpose(1, 2).contiguous().reshape(bsz, q_len, -1)
        attn_output = self.o_proj(attn_output)

        return attn_output, None

    return patched_forward

import qwen3_custom_components
qwen3_custom_components.create_patched_qwen3_attention = create_patched_qwen3_attention_optimized
globals()["create_patched_qwen3_attention"] = create_patched_qwen3_attention_optimized


def patch_model_for_custom_cache(model, config_code: str):
    """Attach a position-aware cache and eviction to the model.

    Returns (cache, eviction_fn, patched_model).

    C4 is intentionally blocked unless a separately validated, source-faithful
    compound adapter is supplied.
    """
    result = qwen3_position_aware_cache_factory(config_code, model)
    cache, eviction_fn = result
    patched_fn = create_patched_qwen3_attention(cache, eviction_fn, force_eager=True)
    from transformers.models.qwen3.modeling_qwen3 import Qwen3Attention

    # Save the original unpatched forward from class
    orig_forward = Qwen3Attention.forward

    for module in model.modules():
        if isinstance(module, Qwen3Attention):
            # Wrap the patched function with our robust interface
            robust_fn = make_robust_attention_forward(orig_forward, patched_fn, cache)
            module.forward = robust_fn.__get__(module, Qwen3Attention)
    return cache, eviction_fn

def validate_full_cache(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 999)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    cache = full_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    assert output.past_key_values is cache
    assert int(cache.get_seq_length()) == 512
    live = tensor_storage_bytes(cache)
    assert live > 0
    return {"actual_method": "FullKV", "cache_live_bytes": live, "seq_length": 512}

def validate_custom_cache_noop(model, cache, reference_logits: torch.Tensor) -> dict[str, Any]:
    """Gate 1: custom-cache logits and greedy token match unpatched C1."""
    inputs = make_niah_input(512, 0.5, 997)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    max_logit_error = (output.logits.float() - reference_logits.float()).abs().max().item()
    if max_logit_error > 1e-1:
        raise AssertionError(f"Custom-cache logits diverged from eager C1 (max error {max_logit_error:.6f}).")
    if not torch.equal(output.logits[:, -1, :].argmax(dim=-1), reference_logits[:, -1, :].argmax(dim=-1)):
        raise AssertionError("Custom-cache greedy continuation differs from eager C1.")
    assert output.past_key_values is cache
    assert cache.get_seq_length() == 512
    for layer_idx in range(cache.num_layers):
        k, v, pos = cache.get_layer(layer_idx)
        assert k is not None and v is not None, f"Layer {layer_idx} missing cache"
        assert k.shape[2] == 512, f"Layer {layer_idx}: expected 512 tokens, got {k.shape[2]}"
        assert pos.shape[0] == 512 and pos[-1].item() == 511
    live = tensor_storage_bytes(cache)
    return {
        "actual_method": "FullKV (custom cache)", "cache_live_bytes": live,
        "seq_length": 512, "max_logit_error": max_logit_error,
    }

def validate_turboquant_or_fallback(model) -> dict[str, Any]:
    inputs = make_niah_input(512, 0.5, 998)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    errors = []
    try:
        cache = turboquant_cache(model)
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
        assert output.past_key_values is cache
        assert int(cache.get_seq_length()) == 512
        assert len(cache.layers) == model.config.num_hidden_layers
        layer = cache.layers[0]
        assert layer._key_indices.dtype == torch.uint8
        assert layer._key_indices.shape[-1] == model.config.head_dim // 2
        assert layer._value_indices.shape[-1] == model.config.head_dim // 2
        live = tensor_storage_bytes(cache)
        reported = cache.memory_usage_bytes()
        if live <= 0 or reported["compressed_bytes"] <= 0:
            raise AssertionError("TurboQuant did not expose non-empty compressed storage.")
        fp16_cache_bytes = (
            model.config.num_hidden_layers
            * model.config.num_key_value_heads
            * 512
            * model.config.head_dim
            * 2
            * 2
        )
        if live >= fp16_cache_bytes:
            raise AssertionError(
                f"TurboQuant compressed indices exist, but reachable live cache storage "
                f"({live / 2**20:.1f} MiB) is not below the FP16 equivalent "
                f"({fp16_cache_bytes / 2**20:.1f} MiB)."
            )
        return {
            "actual_method": "TurboQuant-community-4bit",
            "cache_factory": turboquant_cache,
            "cache_live_bytes": live,
            "reported": reported,
            "errors": errors,
        }
    except Exception as exc:
        errors.append(f"TurboQuant rejected: {type(exc).__name__}: {exc}")
        clear_cuda()

    cache = hqq_cache(model)
    with torch.inference_mode():
        output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
    assert output.past_key_values is cache
    assert int(cache.get_seq_length()) == 512
    live = tensor_storage_bytes(cache)
    if live <= 0:
        raise AssertionError("HQQ fallback did not expose cache storage.")
    fp16_cache_bytes = (
        model.config.num_hidden_layers
        * model.config.num_key_value_heads
        * 512
        * model.config.head_dim
        * 2
        * 2
    )
    if live >= fp16_cache_bytes:
        raise AssertionError("HQQ fallback did not reduce reachable storage below the FP16 cache.")
    return {
        "actual_method": "HF_HQQ_4BIT_KIVI_INSPIRED",
        "cache_factory": hqq_cache,
        "cache_live_bytes": live,
        "errors": errors,
    }

# --- Run pilots ---
# C0 / C1 with standard DynamicCache
pilot_rows = []
bf16_model = load_model(MODEL_BF16)
pilot_rows.append({"config": "C0", **validate_full_cache(bf16_model)})
del bf16_model
clear_cuda()

awq_model = load_model(MODEL_AWQ)
pilot_rows.append({"config": "C1", **validate_full_cache(awq_model)})

# C1_patched: custom cache with no-op eviction (equivalence gate)
noop_inputs = make_niah_input(512, 0.5, 997)
noop_inputs = {key: value.to(awq_model.device) for key, value in noop_inputs.items()}
with torch.inference_mode():
    noop_reference = awq_model(
        **noop_inputs, past_key_values=full_cache(awq_model), use_cache=True, logits_to_keep=1
    ).logits.detach()
sniff_original_forward(awq_model)
cache_c1p, _ = patch_model_for_custom_cache(awq_model, "C1_patched")
pilot_rows.append({"config": "C1_patched", **validate_custom_cache_noop(awq_model, cache_c1p, noop_reference)})
# Restore for next pilot
restore_original_forward(awq_model)

# C2 accepts only the requested, physically validated TurboQuant method.
# A working HQQ path is preserved as a distinct exploratory configuration.
c2_pilot = validate_turboquant_or_fallback(awq_model)
pilot_rows.append({key: value for key, value in {"config": "C2", **c2_pilot}.items() if key != "cache_factory"})
if c2_pilot["actual_method"] == "TurboQuant-community-4bit":
    CONFIGS["C2"] = ExperimentConfig(
        "C2", MODEL_AWQ, "TurboQuant-4bit", turboquant_cache,
        "TurboQuant-community-4bit", True,
    )
else:
    CONFIGS["C2"] = ExperimentConfig(
        "C2", MODEL_AWQ, "TurboQuant-4bit", None, "BLOCKED", False,
        "; ".join(c2_pilot.get("errors", [])),
    )
    CONFIGS["C2_HQQ"] = ExperimentConfig(
        "C2_HQQ", MODEL_AWQ, "HQQ-4bit exploratory", hqq_cache,
        "HF_HQQ_4BIT_KIVI_INSPIRED", True,
        "Exploratory configuration; not TurboQuant.",
    )

# C3_snapkv: SnapKV with uniform budget at 512 (no-op)
sniff_original_forward(awq_model)
cache_c3s, _ = patch_model_for_custom_cache(awq_model, "C3_snapkv")
pilot_rows.append({"config": "C3_snapkv", **validate_custom_cache_noop(awq_model, cache_c3s, noop_reference)})
restore_original_forward(awq_model)

del awq_model
clear_cuda()

(ROOT / "cache_pilots.json").write_text(json.dumps(pilot_rows, indent=2, default=str), encoding="utf-8")
display(pd.DataFrame(pilot_rows))
display(display_capabilities())


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1164: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/tmp/ipykernel_1119/3069378758.py:36: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
/usr/local/lib/python3.12/dist-packages/torch/_inductor/ops_handler.py:789: UserWarning: undefined OpHandler.data, please add missing op schema
  warnings.warn(f"undefined OpHandler.{name}, please add missing op schema")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

INFO  Kernel: Auto-selection: adding candidate `AwqGEMMTritonLinear`           


INFO  Kernel: selected -> `AwqGEMMTritonLinear`.                               


Loading weights:   0%|          | 0/902 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6132 objects in 0.336s                            


AssertionError: Custom-cache logits diverged from eager C1 (max error 0.109375).

## 6c. Diagnostic layerwise-selection gate

Verify the local diagnostic layerwise budget implementation: earlier layers
have fewer tokens, later layers have more, and the total is 36,864 entries.
This checks position/cache mechanics only. It cannot validate a PyramidKV
claim without a pinned, inspected reference allocation.

In [ ]:
PYRAMIDKV_GATE_RESULTS = []

for ctx in (512, 2048, 8192):
    awq_model = load_model(MODEL_AWQ)
    sniff_original_forward(awq_model)
    cache_c3, _ = patch_model_for_custom_cache(awq_model, "C3")
    inputs = make_niah_input(ctx, 0.5, 40000 + ctx)
    inputs_gpu = {key: value.to(awq_model.device) for key, value in inputs.items()}
    with torch.inference_mode():
        output = awq_model(**inputs_gpu, past_key_values=cache_c3, use_cache=True, logits_to_keep=1)
    # Decode a few steps
    next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
    for _ in range(16):
        next_token = awq_model(input_ids=next_token, past_key_values=cache_c3, use_cache=True, logits_to_keep=1).logits[:, -1, :].argmax(dim=-1, keepdim=True)

    gate = {"context": ctx, "pass": True, "errors": [], "layer_retained": []}
    total_retained = 0
    for layer_idx in range(cache_c3.num_layers):
        k, v, pos = cache_c3.get_layer(layer_idx)
        if k is None:
            gate["errors"].append(f"Layer {layer_idx} missing cache")
            gate["pass"] = False
            continue
        retained = k.shape[2]
        gate["layer_retained"].append(retained)
        total_retained += retained
        budget = CUSTOM_LAYERWISE_BUDGETS[layer_idx]
        effective_cap = min(budget, cache_c3.logical_length(layer_idx))
        if retained > effective_cap:
            gate["errors"].append(
                f"L{layer_idx}: retained {retained} > budget {effective_cap}"
            )
            gate["pass"] = False
    gate["total_retained"] = total_retained
    PYRAMIDKV_GATE_RESULTS.append(gate)
    restore_original_forward(awq_model)
    del awq_model
    clear_cuda()

display(pd.DataFrame(PYRAMIDKV_GATE_RESULTS))
if all(r["pass"] for r in PYRAMIDKV_GATE_RESULTS):
    print("Diagnostic layerwise-selection gate passed; C3 remains blocked pending a pinned PyramidKV source audit.")
else:
    print("Diagnostic layerwise-selection gate failed; C3 remains blocked.")

## 6d. TurboQuant standalone round-trip audit

TurboQuant is audited outside the model first:
1. Round-trip on Qwen3-shaped K/V tensors at 512 and 8192.
2. Check packed storage, finite reconstruction, no persistent FP16 duplicate.
3. Multi-step cache test: prefill + 32 token-by-token updates.

Passing this audit can validate C2 only. C4 remains blocked because there is
no source-faithful Qwen3 PyramidKV adapter to compose with TurboQuant.

In [ ]:
def audit_turboquant_outside_model() -> dict[str, Any]:
    """Standalone TurboQuant audit on Qwen3-shaped tensors."""
    results = {"pass": True, "errors": [], "compress_ratio": []}
    # This must be completed from an inspected, pinned source before C2 can
    # use the TurboQuant name. A passing storage test alone is insufficient.
    turboquant_method_manifest = {
        "repository": "back2matching/turboquant",
        "commit": TURBOQUANT_COMMIT,
        "source_audited": False,
        "audit_note": "Set source_audited=True only after recording rotation, quantizer, K/V format, and dequantization-path evidence.",
    }
    try:
        from turboquant import TurboQuantCache
    except ImportError as e:
        results["pass"] = False
        results["errors"].append(f"TurboQuant import failed: {e}")
        return results

    # Test at two sequence lengths
    for n_tokens in (512, 8192):
        # Create Qwen3-shaped K/V: [1, 8, n_tokens, 128]
        k = torch.randn(1, 8, n_tokens, 128, dtype=torch.float16, device="cuda")
        v = torch.randn(1, 8, n_tokens, 128, dtype=torch.float16, device="cuda")

        cache = TurboQuantCache(bits=4)
        try:
            # Simulate prefill: store these K/V
            with torch.inference_mode():
                cache.update(k, v, 0, {"cache_position": torch.arange(n_tokens, device="cuda")})

            live_bytes = tensor_storage_bytes(cache)
            fp16_bytes = k.numel() * 2 * 2  # K + V in FP16

            if live_bytes <= 0:
                results["errors"].append(f"{n_tokens}t: cache exposes 0 bytes")
                results["pass"] = False
            if live_bytes >= fp16_bytes:
                results["errors"].append(
                    f"{n_tokens}t: live bytes {live_bytes} >= FP16 {fp16_bytes}"
                )
                results["pass"] = False

            ratio = fp16_bytes / max(live_bytes, 1)
            results["compress_ratio"].append({"tokens": n_tokens, "fp16_bytes": fp16_bytes, "compressed_bytes": live_bytes, "ratio": ratio})

            # Reject a full-length persistent FP16 K/V history. A bounded
            # residual window is acceptable only when it is shorter than the
            # logical sequence length.
            seen = set()
            fp16_history = []
            def walk(value, path="cache"):
                if id(value) in seen:
                    return
                seen.add(id(value))
                if isinstance(value, torch.Tensor):
                    if value.dtype in (torch.float16, torch.bfloat16) and value.ndim >= 3 and value.shape[-2] >= n_tokens:
                        fp16_history.append((path, tuple(value.shape)))
                elif isinstance(value, dict):
                    for key, child in value.items():
                        walk(child, f"{path}.{key}")
                elif isinstance(value, (list, tuple)):
                    for index, child in enumerate(value):
                        walk(child, f"{path}[{index}]")
                elif hasattr(value, "__dict__"):
                    for key, child in vars(value).items():
                        walk(child, f"{path}.{key}")
            walk(cache)
            if fp16_history:
                results["errors"].append(f"{n_tokens}t: persistent full-length FP16 history: {fp16_history[:2]}")
                results["pass"] = False

        except Exception as e:
            results["errors"].append(f"{n_tokens}t: {type(e).__name__}: {e}")
            results["pass"] = False

        # Multi-step decode test
        try:
            cache2 = TurboQuantCache(bits=4)
            with torch.inference_mode():
                cache2.update(k[:, :, :1, :], v[:, :, :1, :], 0, {"cache_position": torch.tensor([0], device="cuda")})
            for step in range(1, 32):
                new_k = torch.randn(1, 8, 1, 128, dtype=torch.float16, device="cuda")
                new_v = torch.randn(1, 8, 1, 128, dtype=torch.float16, device="cuda")
                with torch.inference_mode():
                    cache2.update(new_k, new_v, 0, {"cache_position": torch.tensor([step], device="cuda")})
            seq_len = cache2.get_seq_length()
            if seq_len != 32:
                results["errors"].append(f"Multi-step decode: seq_len={seq_len}, expected 32")
                results["pass"] = False
        except Exception as e:
            results["errors"].append(f"Multi-step decode: {type(e).__name__}: {e}")
            results["pass"] = False

        del cache, cache2
        torch.cuda.empty_cache()

    if not turboquant_method_manifest["source_audited"]:
        results["errors"].append("TurboQuant source-method audit is not recorded; C2 remains blocked.")
        results["pass"] = False
    results["method_manifest"] = turboquant_method_manifest
    return results

TQ_AUDIT = audit_turboquant_outside_model()
display(pd.DataFrame([TQ_AUDIT]))

if TQ_AUDIT["pass"]:
    CONFIGS["C2"] = ExperimentConfig(
        "C2", MODEL_AWQ, "TurboQuant-4bit", turboquant_cache,
        "TurboQuant-community-4bit", True, "",
    )
    print("C2 TurboQuant VALIDATED")
else:
    # HQQ remains explicitly separate from the requested TurboQuant C2.
    CONFIGS["C2"] = ExperimentConfig(
        "C2", MODEL_AWQ, "TurboQuant-4bit", None,
        "BLOCKED", False,
        f"TurboQuant failed: {'; '.join(TQ_AUDIT['errors'][:3])}",
    )
    CONFIGS["C2_HQQ"] = ExperimentConfig(
        "C2_HQQ", MODEL_AWQ, "HQQ-4bit exploratory", hqq_cache,
        "HF_HQQ_4BIT_KIVI_INSPIRED", True,
        "Exploratory configuration; not TurboQuant.",
    )
    print("C2 TurboQuant BLOCKED; C2_HQQ is available as a separately labelled exploratory configuration.")

(ROOT / "turboquant_audit.json").write_text(json.dumps(TQ_AUDIT, indent=2, default=str), encoding="utf-8")

## 7. Timed prefill and decode runner

Performance runs use fixed 128-token greedy decode. NIAH uses a separate short
completion workload. Model loading and prompt construction are outside the measured
interval. The runner records actual output-token counts rather than configured maxima.

In [ ]:
def wait_for_cooldown(target_c: int = COOLDOWN_TARGET_C, timeout_s: int = COOLDOWN_TIMEOUT_S) -> float:
    start = time.monotonic()
    while time.monotonic() - start < timeout_s:
        temperature = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        if temperature <= target_c:
            return float(temperature)
        time.sleep(2)
    return float(pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU))

def decode_response(token_ids: torch.Tensor) -> str:
    return tokenizer.decode(token_ids.tolist(), skip_special_tokens=True, clean_up_tokenization_spaces=False)

def measured_generate(
    model,
    cfg: ExperimentConfig,
    inputs_cpu: dict[str, torch.Tensor],
    max_new_tokens: int,
    trace_stem: str,
) -> dict[str, Any]:
    if cfg.cache_factory is None or not cfg.method_validated:
        raise RuntimeError(f"{cfg.code} is not executable: {cfg.blocked_reason}")
    wait_for_cooldown()
    clear_cuda()
    inputs = {key: value.to(model.device) for key, value in inputs_cpu.items()}
    cache = cfg.cache_factory(model)
    loaded_mb = torch.cuda.memory_allocated() / 2**20
    monitor = GPUMonitor()
    monitor.start()
    started = time.perf_counter()
    try:
        prefill_started = time.perf_counter()
        with torch.inference_mode():
            output = model(**inputs, past_key_values=cache, use_cache=True, logits_to_keep=1)
        torch.cuda.synchronize()
        prefill_s = time.perf_counter() - prefill_started

        next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = []
        attention_mask = inputs["attention_mask"]
        decode_started = time.perf_counter()
        for _ in range(max_new_tokens):
            generated.append(next_token)
            if int(next_token.item()) in {tokenizer.eos_token_id, 151645}:
                break
            attention_mask = torch.cat(
                [attention_mask, torch.ones((attention_mask.shape[0], 1), dtype=attention_mask.dtype, device=model.device)],
                dim=1,
            )
            with torch.inference_mode():
                output = model(
                    input_ids=next_token,
                    attention_mask=attention_mask,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
            next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        torch.cuda.synchronize()
        decode_s = time.perf_counter() - decode_started
        total_s = time.perf_counter() - started
        power = monitor.stop(IDLE_POWER_W)
    except Exception:
        with contextlib.suppress(Exception):
            monitor.stop(IDLE_POWER_W)
        raise

    output_ids = torch.cat(generated, dim=1)[0] if generated else torch.empty(0, dtype=torch.long, device=model.device)
    response = decode_response(output_ids.detach().cpu())
    cache_live_mb = tensor_storage_bytes(cache) / 2**20
    trace_path = TRACE_DIR / f"{trace_stem}.jsonl"
    with trace_path.open("w", encoding="utf-8") as handle:
        for sample in monitor.samples:
            handle.write(json.dumps(sample) + "\n")
    output_tokens = int(output_ids.numel())
    return {
        "response": response,
        "input_tokens": int(inputs["input_ids"].shape[1]),
        "output_tokens": output_tokens,
        "prefill_s": prefill_s,
        "decode_s": decode_s,
        "total_s": total_s,
        "prefill_tok_s": int(inputs["input_ids"].shape[1]) / max(prefill_s, 1e-9),
        "decode_tok_s": output_tokens / max(decode_s, 1e-9),
        "torch_loaded_mb": loaded_mb,
        "torch_peak_mb": torch.cuda.max_memory_allocated() / 2**20,
        "cache_live_mb": cache_live_mb,
        "trace_file": str(trace_path.relative_to(ROOT)),
        "energy_per_output_token_j": power["gross_energy_j"] / max(output_tokens, 1),
        **power,
    }

def base_result(cfg: ExperimentConfig, experiment: str, context: int, **extra) -> dict[str, Any]:
    return {
        "run_id": str(uuid.uuid4()),
        "timestamp_utc": utc_now(),
        "experiment": experiment,
        "config": cfg.code,
        "requested_method": cfg.requested_method,
        "actual_method": cfg.actual_method,
        "method_validated": cfg.method_validated,
        "model_id": cfg.model_id,
        "context_target": context,
        "idle_power_w": IDLE_POWER_W,
        "valid": False,
        "error": "",
        **extra,
    }

def run_and_save(model, cfg: ExperimentConfig, inputs, output_path: Path, trace_stem: str, **metadata):
    row = base_result(cfg, metadata.pop("experiment"), metadata.pop("context_target"), **metadata)
    try:
        row.update(measured_generate(model, cfg, inputs, metadata.get("max_new_tokens", DECODE_TOKENS), trace_stem))
        row["valid"] = (
            row["power_samples"] >= 2
            and 5.0 <= row["min_power_w"] <= row["max_power_w"] <= 100.0
            and row["input_tokens"] == row["context_target"]
        )
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
    append_csv_atomic(output_path, row)
    return row

## 8. Mandatory sanity checks

C0 must retrieve the needle at 512 tokens. C1 and each validated cache
configuration are then checked. Stop if C0 fails.

In [ ]:
SANITY_CSV = RAW_DIR / "sanity.csv"
sanity_rows = []
for config_code in ("C0", "C1", "C2", "C2_HQQ"):
    if config_code not in CONFIGS:
        continue
    cfg = CONFIGS[config_code]
    if not cfg.method_validated and config_code != "C2":
        sanity_rows.append({"config": config_code, "correct": False, "valid": False, "error": cfg.blocked_reason})
        continue
    model = load_model(cfg.model_id)
    # For C3_snapkv, patch before generating
    if cfg.custom_cache:
        sniff_original_forward(model)
        model_cache, _ = patch_model_for_custom_cache(model, config_code)
        cache = model_cache
    else:
        cache = None
    inputs = make_niah_input(512, 0.5, 10000 + len(sanity_rows))
    # For custom cache, use it directly
    if cfg.custom_cache:
        row = {}
        try:
            wait_for_cooldown()
            clear_cuda()
            inputs_gpu = {key: value.to(model.device) for key, value in inputs.items()}
            monitor = GPUMonitor()
            monitor.start()
            with torch.inference_mode():
                output = model(**inputs_gpu, past_key_values=cache, use_cache=True, logits_to_keep=1)
            response_ids = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            for _ in range(NIAH_MAX_NEW_TOKENS - 1):
                response_ids = torch.cat([response_ids, model(input_ids=response_ids[:, -1:], past_key_values=cache, use_cache=True, logits_to_keep=1).logits[:, -1, :].argmax(dim=-1, keepdim=True)], dim=1)
            response = decode_response(response_ids.detach().cpu())
            row["response"] = response
            row["correct"] = niah_correct(response)
            row["input_tokens"] = int(inputs["input_ids"].shape[1])
            row["output_tokens"] = int(response_ids.shape[1])
            row["valid"] = True
            row["actual_method"] = cfg.actual_method
            row["config"] = config_code
            power = monitor.stop(IDLE_POWER_W)
            row.update(power)
            append_csv_atomic(SANITY_CSV, row)
        except Exception as exc:
            row = {"config": config_code, "correct": False, "valid": False, "error": f"{type(exc).__name__}: {exc}"}
        sanity_rows.append(row)
        restore_original_forward(model)
    else:
        row = run_and_save(
            model, cfg, inputs, SANITY_CSV, f"sanity_{config_code}",
            experiment="sanity", context_target=512, block=-1, item_id="sanity",
            thinking=False, needle_depth=0.5, max_new_tokens=NIAH_MAX_NEW_TOKENS,
        )
        row["correct"] = niah_correct(row.get("response", ""))
        sanity_rows.append(row)
    del model
    clear_cuda()

display(pd.DataFrame(sanity_rows)[["config", "actual_method", "input_tokens", "output_tokens", "correct", "valid", "error"]])
if not sanity_rows[0].get("correct"):
    raise RuntimeError("C0 failed 512-token NIAH. Fix the prompt/evaluator before collecting data.")
if not all(row.get("valid") or row.get("config") == "C3_snapkv" for row in sanity_rows):
    raise RuntimeError("At least one sanity run failed measurement validation.")

## 9. Full performance matrix (restart-safe)

Only validated configurations run. C0 loads BF16. C1/C2/C3_snapkv/C3/C4
are randomized within each AWQ block. Blocked configs emit a blocked row
rather than being silently skipped.

In [ ]:
PERF_CSV = RAW_DIR / "performance.csv"

def completed_keys(path: Path, experiment: str) -> set[tuple]:
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    frame = frame[(frame["experiment"] == experiment) & (frame["valid"] == True)]
    return set(zip(frame["config"], frame["context_target"], frame["block"], strict=False))

def run_performance_for_model(model_id: str, config_codes: list[str]) -> None:
    done = completed_keys(PERF_CSV, "performance")
    model = load_model(model_id)
    try:
        for block in range(PERFORMANCE_BLOCKS):
            order = list(config_codes)
            random.Random(SEED + block).shuffle(order)
            for context in CONTEXT_LENGTHS:
                inputs = make_niah_input(context, 0.5, 20000 + block)
                for code in order:
                    if (code, context, block) in done:
                        continue
                    cfg = CONFIGS[code]
                    if not cfg.method_validated:
                        row = base_result(cfg, "performance", context, block=block, item_id=f"perf_{block:02d}", thinking=False, needle_depth=0.5)
                        row["error"] = f"BLOCKED: {cfg.blocked_reason}"
                        row["valid"] = False
                        append_csv_atomic(PERF_CSV, row)
                        print(code, context, block, "BLOCKED", cfg.blocked_reason)
                        continue
                    if cfg.custom_cache:
                        sniff_original_forward(model)
                        model_cache, _ = patch_model_for_custom_cache(model, code)
                        wait_for_cooldown()
                        clear_cuda()
                        inputs_gpu = {key: value.to(model.device) for key, value in inputs.items()}
                        monitor = GPUMonitor()
                        monitor.start()
                        try:
                            prefill_start = time.perf_counter()
                            loaded_mb = torch.cuda.memory_allocated() / 2**20
                            with torch.inference_mode():
                                output = model(**inputs_gpu, past_key_values=model_cache, use_cache=True, logits_to_keep=1)
                            torch.cuda.synchronize()
                            prefill_s = time.perf_counter() - prefill_start

                            # Greedy decode 128 tokens
                            generated = []
                            next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                            decode_start = time.perf_counter()
                            for _ in range(DECODE_TOKENS):
                                generated.append(next_token)
                                if int(next_token.item()) in {tokenizer.eos_token_id, 151645}:
                                    break
                                next_token = model(input_ids=next_token, past_key_values=model_cache, use_cache=True, logits_to_keep=1).logits[:, -1, :].argmax(dim=-1, keepdim=True)
                            torch.cuda.synchronize()
                            decode_s = time.perf_counter() - decode_start
                            total_s = time.perf_counter() - prefill_start
                            power = monitor.stop(IDLE_POWER_W)
                            output_ids = torch.cat(generated, dim=1)[0] if generated else torch.empty(0, dtype=torch.long, device=model.device)
                            response = decode_response(output_ids.detach().cpu())
                            cache_live_mb = tensor_storage_bytes(model_cache) / 2**20
                            row = base_result(cfg, "performance", context, block=block, item_id=f"perf_{block:02d}", thinking=False, needle_depth=0.5)
                            row.update({
                                "response": response, "input_tokens": context,
                                "output_tokens": int(output_ids.numel()) if output_ids.numel() > 0 else 0,
                                "prefill_s": prefill_s, "decode_s": decode_s, "total_s": total_s,
                                "prefill_tok_s": context / max(prefill_s, 1e-9),
                                "decode_tok_s": int(output_ids.numel() or 1) / max(decode_s, 1e-9),
                                "torch_loaded_mb": loaded_mb,
                                "torch_peak_mb": torch.cuda.max_memory_allocated() / 2**20,
                                "cache_live_mb": cache_live_mb,
                                "energy_per_output_token_j": power["gross_energy_j"] / max(int(output_ids.numel() or 1), 1),
                                **power,
                            })
                            row["valid"] = power["power_samples"] >= 2 and 5.0 <= power["min_power_w"] <= power["max_power_w"] <= 100.0 and row["input_tokens"] == context
                            append_csv_atomic(PERF_CSV, row)
                            print(code, context, block, row["valid"], row.get("gross_energy_j"))
                        except Exception as exc:
                            row = base_result(cfg, "performance", context, block=block, item_id=f"perf_{block:02d}", thinking=False, needle_depth=0.5)
                            row["error"] = f"{type(exc).__name__}: {exc}"
                            append_csv_atomic(PERF_CSV, row)
                            traceback.print_exc()
                        restore_original_forward(model)
                    else:
                        row = run_and_save(
                            model, cfg, inputs, PERF_CSV, f"perf_{code}_{context}_b{block:02d}",
                            experiment="performance", context_target=context, block=block,
                            item_id=f"perf_{block:02d}", thinking=False, needle_depth=0.5,
                            max_new_tokens=DECODE_TOKENS,
                        )
                    print(code, context, block, row["valid"], row.get("gross_energy_j"), row.get("error", ""))
    finally:
        del model
        clear_cuda()

if RUN_FULL_PERFORMANCE:
    run_performance_for_model(MODEL_BF16, ["C0"])
    perf_codes = ["C1"]
    for code in ("C2", "C2_HQQ", "C3_snapkv", "C3", "C4"):
        if code in CONFIGS and CONFIGS[code].method_validated:
            perf_codes.append(code)
    run_performance_for_model(MODEL_AWQ, perf_codes)
else:
    print("Performance matrix skipped. Set RUN_FULL_PERFORMANCE=True in Section 2.")

## 10. NIAH retrieval matrix

Thirty matched items per condition are balanced over 25%, 50%, and 75% needle depth.

In [ ]:
NIAH_CSV = RAW_DIR / "niah.csv"

def completed_items(path: Path, experiment: str) -> set[tuple]:
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    frame = frame[(frame["experiment"] == experiment) & (frame["valid"] == True)]
    return set(zip(frame["config"], frame["context_target"], frame["item_id"], strict=False))

def run_niah_for_model(model_id: str, config_codes: list[str]) -> None:
    done = completed_items(RAW_DIR / "niah_scores.csv", "niah")
    model = load_model(model_id)
    depths = (0.25, 0.50, 0.75)
    try:
        for context in CONTEXT_LENGTHS:
            for item_index in range(NIAH_ITEMS_PER_CONDITION):
                item_id = f"niah_{item_index:02d}"
                depth = depths[item_index % len(depths)]
                inputs = make_niah_input(context, depth, item_index)
                order = list(config_codes)
                random.Random(SEED + context + item_index).shuffle(order)
                for code in order:
                    if (code, context, item_id) in done:
                        continue
                    cfg = CONFIGS[code]
                    if not cfg.method_validated:
                        row = base_result(cfg, "niah", context, block=item_index, item_id=item_id, thinking=False, needle_depth=depth)
                        row["error"] = f"BLOCKED: {cfg.blocked_reason}"
                        append_csv_atomic(NIAH_CSV, row)
                        continue
                    if cfg.custom_cache:
                        sniff_original_forward(model)
                        model_cache, _ = patch_model_for_custom_cache(model, code)
                        try:
                            wait_for_cooldown()
                            clear_cuda()
                            inputs_gpu = {key: value.to(model.device) for key, value in inputs.items()}
                            monitor = GPUMonitor()
                            monitor.start()
                            prefill_start = time.perf_counter()
                            loaded_mb = torch.cuda.memory_allocated() / 2**20
                            with torch.inference_mode():
                                output = model(**inputs_gpu, past_key_values=model_cache, use_cache=True, logits_to_keep=1)
                            torch.cuda.synchronize()
                            prefill_s = time.perf_counter() - prefill_start
                            generated = []
                            next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                            decode_start = time.perf_counter()
                            for _step in range(NIAH_MAX_NEW_TOKENS):
                                generated.append(next_token)
                                if int(next_token.item()) in {tokenizer.eos_token_id, 151645}:
                                    break
                                next_token = model(input_ids=next_token, past_key_values=model_cache, use_cache=True, logits_to_keep=1).logits[:, -1, :].argmax(dim=-1, keepdim=True)
                            torch.cuda.synchronize()
                            decode_s = time.perf_counter() - decode_start
                            total_s = time.perf_counter() - prefill_start
                            power = monitor.stop(IDLE_POWER_W)
                            output_ids = torch.cat(generated, dim=1)[0]
                            response = decode_response(output_ids.detach().cpu())
                            cache_live_mb = tensor_storage_bytes(model_cache) / 2**20
                            row = base_result(cfg, "niah", context, block=item_index, item_id=item_id, thinking=False, needle_depth=depth)
                            row.update({
                                "response": response, "input_tokens": context,
                                "output_tokens": int(output_ids.numel()),
                                "prefill_s": prefill_s, "decode_s": decode_s, "total_s": total_s,
                                "prefill_tok_s": context / max(prefill_s, 1e-9),
                                "decode_tok_s": int(output_ids.numel()) / max(decode_s, 1e-9),
                                "torch_loaded_mb": loaded_mb,
                                "torch_peak_mb": torch.cuda.max_memory_allocated() / 2**20,
                                "cache_live_mb": cache_live_mb,
                                "energy_per_output_token_j": power["gross_energy_j"] / max(int(output_ids.numel()), 1),
                                **power,
                            })
                            row["valid"] = power["power_samples"] >= 2 and 5.0 <= power["min_power_w"] <= power["max_power_w"] <= 100.0 and row["input_tokens"] == context
                            row["correct"] = niah_correct(row.get("response", ""))
                            append_csv_atomic(NIAH_CSV, row)
                            append_csv_atomic(RAW_DIR / "niah_scores.csv", row)
                            print(code, context, item_id, row["correct"], row["valid"])
                        except Exception as exc:
                            row = base_result(cfg, "niah", context, block=item_index, item_id=item_id, thinking=False, needle_depth=depth)
                            row["error"] = f"{type(exc).__name__}: {exc}"
                            append_csv_atomic(NIAH_CSV, row)
                            traceback.print_exc()
                        finally:
                            restore_original_forward(model)
                    else:
                        row = run_and_save(
                            model, cfg, inputs, NIAH_CSV, f"niah_{code}_{context}_{item_index:02d}",
                            experiment="niah", context_target=context, block=item_index,
                            item_id=item_id, thinking=False, needle_depth=depth,
                            max_new_tokens=NIAH_MAX_NEW_TOKENS,
                        )
                        row["correct"] = niah_correct(row.get("response", ""))
                        append_csv_atomic(RAW_DIR / "niah_scores.csv", row)
                        print(code, context, item_id, row["correct"], row["valid"])
    finally:
        del model
        clear_cuda()

if RUN_FULL_NIAH:
    run_niah_for_model(MODEL_BF16, ["C0"])
    niah_codes = ["C1"]
    for code in ("C2", "C2_HQQ", "C3_snapkv", "C3", "C4"):
        if code in CONFIGS and CONFIGS[code].method_validated:
            niah_codes.append(code)
    run_niah_for_model(MODEL_AWQ, niah_codes)
else:
    print("NIAH matrix skipped. Set RUN_FULL_NIAH=True in Section 2.")

## 11. Thinking-mode exact-answer subset

C4 cannot run until eviction support is validated. The notebook records C0/C1 and
C2 as an informative quantized-cache extension; the capability table documents the
preregistered C4 omission.

In [ ]:
THINK_CSV = RAW_DIR / "thinking.csv"

def run_thinking_for_model(model_id: str, config_codes: list[str]) -> None:
    done = completed_items(RAW_DIR / "thinking_scores.csv", "thinking")
    model = load_model(model_id)
    items = reasoning_dataset()
    try:
        for item_index, item in enumerate(items):
            inputs = make_reasoning_input(item)
            order = list(config_codes)
            random.Random(SEED + item_index).shuffle(order)
            for code in order:
                if (code, 8192, item["item_id"]) in done:
                    continue
                cfg = CONFIGS[code]
                if not cfg.method_validated:
                    row = base_result(cfg, "thinking", 8192, block=item_index, item_id=item["item_id"], thinking=True, needle_depth=np.nan)
                    row["error"] = f"BLOCKED: {cfg.blocked_reason}"
                    append_csv_atomic(THINK_CSV, row)
                    continue
                if cfg.custom_cache:
                    sniff_original_forward(model)
                    model_cache, _ = patch_model_for_custom_cache(model, code)
                    try:
                        wait_for_cooldown()
                        clear_cuda()
                        inputs_gpu = {key: value.to(model.device) for key, value in inputs.items()}
                        monitor = GPUMonitor()
                        monitor.start()
                        prefill_start = time.perf_counter()
                        loaded_mb = torch.cuda.memory_allocated() / 2**20
                        with torch.inference_mode():
                            output = model(**inputs_gpu, past_key_values=model_cache, use_cache=True, logits_to_keep=1)
                        torch.cuda.synchronize()
                        prefill_s = time.perf_counter() - prefill_start
                        generated = []
                        next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                        decode_start = time.perf_counter()
                        for _step in range(THINKING_MAX_NEW_TOKENS):
                            generated.append(next_token)
                            if int(next_token.item()) in {tokenizer.eos_token_id, 151645}:
                                break
                            next_token = model(input_ids=next_token, past_key_values=model_cache, use_cache=True, logits_to_keep=1).logits[:, -1, :].argmax(dim=-1, keepdim=True)
                        torch.cuda.synchronize()
                        decode_s = time.perf_counter() - decode_start
                        total_s = time.perf_counter() - prefill_start
                        power = monitor.stop(IDLE_POWER_W)
                        output_ids = torch.cat(generated, dim=1)[0] if generated else torch.empty(0, dtype=torch.long, device=model.device)
                        response = decode_response(output_ids.detach().cpu())
                        cache_live_mb = tensor_storage_bytes(model_cache) / 2**20
                        row = base_result(cfg, "thinking", 8192, block=item_index, item_id=item["item_id"], thinking=True, needle_depth=np.nan)
                        row.update({
                            "response": response, "input_tokens": 8192,
                            "output_tokens": int(output_ids.numel()) if output_ids.numel() > 0 else 0,
                            "prefill_s": prefill_s, "decode_s": decode_s, "total_s": total_s,
                            "prefill_tok_s": 8192 / max(prefill_s, 1e-9),
                            "decode_tok_s": int(output_ids.numel() or 1) / max(decode_s, 1e-9),
                            "torch_loaded_mb": loaded_mb,
                            "torch_peak_mb": torch.cuda.max_memory_allocated() / 2**20,
                            "cache_live_mb": cache_live_mb,
                            "energy_per_output_token_j": power["gross_energy_j"] / max(int(output_ids.numel() or 1), 1),
                            **power,
                        })
                        row["valid"] = power["power_samples"] >= 2 and 5.0 <= power["min_power_w"] <= power["max_power_w"] <= 100.0 and row["input_tokens"] == 8192
                        predicted = extract_boxed_integer(row.get("response", ""))
                        row["correct"] = predicted == item["answer"]
                        row["expected_answer"] = item["answer"]
                        row["predicted_answer"] = predicted
                        append_csv_atomic(THINK_CSV, row)
                        append_csv_atomic(
                            RAW_DIR / "thinking_scores.csv", row,
                            columns=RESULT_COLUMNS + ["expected_answer", "predicted_answer"],
                        )
                        print(code, item["item_id"], predicted, item["answer"], row["correct"], row["valid"])
                    except Exception as exc:
                        row = base_result(cfg, "thinking", 8192, block=item_index, item_id=item["item_id"], thinking=True, needle_depth=np.nan)
                        row["error"] = f"{type(exc).__name__}: {exc}"
                        append_csv_atomic(THINK_CSV, row)
                        traceback.print_exc()
                    finally:
                        restore_original_forward(model)
                else:
                    row = run_and_save(
                        model, cfg, inputs, THINK_CSV, f"think_{code}_{item_index:02d}",
                        experiment="thinking", context_target=8192, block=item_index,
                        item_id=item["item_id"], thinking=True, needle_depth=np.nan,
                        max_new_tokens=THINKING_MAX_NEW_TOKENS,
                    )
                    predicted = extract_boxed_integer(row.get("response", ""))
                    row["correct"] = predicted == item["answer"]
                    row["expected_answer"] = item["answer"]
                    row["predicted_answer"] = predicted
                    append_csv_atomic(
                        RAW_DIR / "thinking_scores.csv", row,
                        columns=RESULT_COLUMNS + ["expected_answer", "predicted_answer"],
                    )
                    print(code, item["item_id"], predicted, item["answer"], row["correct"], row["valid"])
    finally:
        del model
        clear_cuda()

if RUN_FULL_THINKING:
    run_thinking_for_model(MODEL_BF16, ["C0"])
    think_codes = ["C1"]
    for code in ("C2", "C2_HQQ", "C3_snapkv", "C3", "C4"):
        if code in CONFIGS and CONFIGS[code].method_validated:
            think_codes.append(code)
    run_thinking_for_model(MODEL_AWQ, think_codes)
else:
    print("Thinking subset skipped. Set RUN_FULL_THINKING=True in Section 2.")

## 12. Optional active-cache perplexity

This computes next-token loss sequentially with the selected cache active. It is
intentionally separate from timed energy runs. Set `RUN_PERPLEXITY = True` after the
main pilots succeed. Results are descriptive because this small fixed sample is not a
full WikiText-2 benchmark.

In [ ]:
RUN_PERPLEXITY = False

def active_cache_perplexity(model, cfg: ExperimentConfig, token_ids: torch.Tensor, chunk_size: int = 128) -> float:
    cache = cfg.cache_factory(model)
    token_ids = token_ids.to(model.device)
    total_nll = 0.0
    total_targets = 0
    with torch.inference_mode():
        for start in range(0, token_ids.shape[1] - 1, chunk_size):
            stop = min(start + chunk_size, token_ids.shape[1] - 1)
            current = token_ids[:, start:stop]
            targets = token_ids[:, start + 1:stop + 1]
            output = model(input_ids=current, past_key_values=cache, use_cache=True, logits_to_keep=0)
            logits = output.logits[:, -targets.shape[1]:, :].float()
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.shape[-1]), targets.reshape(-1), reduction="sum"
            )
            total_nll += float(loss.item())
            total_targets += targets.numel()
    return math.exp(total_nll / total_targets)

if RUN_PERPLEXITY:
    sample = repeated_ids(
        "Energy-efficient computing evaluates useful work relative to measured resource consumption. ",
        2049,
    )
    tokens = torch.tensor([sample], dtype=torch.long)
    ppl_rows = []
    for model_id, codes in ((MODEL_BF16, ["C0"]), (MODEL_AWQ, ["C1", "C2"])):
        model = load_model(model_id)
        for code in codes:
            ppl_rows.append({"config": code, "actual_method": CONFIGS[code].actual_method, "perplexity": active_cache_perplexity(model, CONFIGS[code], tokens)})
        del model
        clear_cuda()
    pd.DataFrame(ppl_rows).to_csv(RAW_DIR / "perplexity.csv", index=False)
    display(pd.DataFrame(ppl_rows))

## 13. Validation, statistics, confidence intervals, and figures

Performance uses blocks as paired units. Friedman and paired Wilcoxon tests are run
only when at least three validated configurations have complete paired blocks;
otherwise pairwise Wilcoxon is reported. Bonferroni correction uses the number of
actually tested pairs. Binary matched outcomes use McNemar tests.

In [ ]:
def load_valid(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    if "valid" in frame:
        frame = frame[frame["valid"].astype(str).str.lower().isin(["true", "1"])]
    return frame

perf = load_valid(PERF_CSV)
niah = load_valid(RAW_DIR / "niah_scores.csv")
thinking = load_valid(RAW_DIR / "thinking_scores.csv")

def mean_ci(values, confidence=0.95):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) < 2:
        return (float(values.mean()) if len(values) else np.nan, np.nan, np.nan)
    mean = float(values.mean())
    sem = st.sem(values)
    low, high = st.t.interval(confidence, len(values) - 1, loc=mean, scale=sem)
    return mean, float(low), float(high)

def binary_ci(successes: int, trials: int, confidence=0.95):
    if trials == 0:
        return np.nan, np.nan
    result = binomtest(successes, trials).proportion_ci(confidence_level=confidence, method="exact")
    return float(result.low), float(result.high)

summary_rows = []
if not perf.empty:
    for (config, actual, context), group in perf.groupby(["config", "actual_method", "context_target"]):
        energy_mean, energy_low, energy_high = mean_ci(group["gross_energy_j"])
        summary_rows.append({
            "config": config, "actual_method": actual, "context": context, "n": len(group),
            "energy_mean_j": energy_mean, "energy_ci_low": energy_low, "energy_ci_high": energy_high,
            "power_mean_w": group["avg_power_w"].mean(), "peak_vram_mean_mb": group["torch_peak_mb"].mean(),
            "decode_tok_s_mean": group["decode_tok_s"].mean(), "cache_live_mean_mb": group["cache_live_mb"].mean(),
        })
perf_summary = pd.DataFrame(summary_rows)
perf_summary.to_csv(ROOT / "performance_summary.csv", index=False)

accuracy_rows = []
for name, frame in (("niah", niah), ("thinking", thinking)):
    if frame.empty:
        continue
    frame["correct"] = frame["correct"].astype(str).str.lower().isin(["true", "1"])
    for keys, group in frame.groupby(["config", "actual_method", "context_target"]):
        successes = int(group["correct"].sum())
        low, high = binary_ci(successes, len(group))
        accuracy_rows.append({
            "experiment": name, "config": keys[0], "actual_method": keys[1], "context": keys[2],
            "successes": successes, "n": len(group), "accuracy": successes / len(group),
            "ci_low": low, "ci_high": high,
        })
accuracy_summary = pd.DataFrame(accuracy_rows)
accuracy_summary.to_csv(ROOT / "accuracy_summary.csv", index=False)

def paired_performance_tests(frame: pd.DataFrame, metric="gross_energy_j") -> pd.DataFrame:
    rows = []
    for context, context_frame in frame.groupby("context_target"):
        pivot = context_frame.pivot_table(index="block", columns="config", values=metric, aggfunc="first").dropna()
        configs = list(pivot.columns)
        pairs = [(configs[i], configs[j]) for i in range(len(configs)) for j in range(i + 1, len(configs))]
        for first, second in pairs:
            differences = pivot[first] - pivot[second]
            if len(differences) < 5 or np.allclose(differences, 0):
                statistic, p_value = np.nan, np.nan
            else:
                statistic, p_value = wilcoxon(pivot[first], pivot[second], alternative="two-sided")
            effect_r = np.nan
            if pd.notna(p_value) and p_value > 0:
                z = st.norm.isf(p_value / 2)
                effect_r = float(z / math.sqrt(len(differences)))
            rows.append({
                "context": context, "metric": metric, "config_a": first, "config_b": second,
                "n_pairs": len(differences), "wilcoxon_w": statistic, "p_raw": p_value,
                "median_difference": differences.median(), "effect_r_abs": effect_r,
            })
    result = pd.DataFrame(rows)
    if not result.empty:
        valid_tests = result["p_raw"].notna().sum()
        result["p_bonferroni"] = np.minimum(result["p_raw"] * max(valid_tests, 1), 1.0)
    return result

performance_tests = paired_performance_tests(perf) if not perf.empty else pd.DataFrame()
performance_tests.to_csv(ROOT / "performance_pairwise_tests.csv", index=False)

def performance_omnibus_tests(frame: pd.DataFrame, metric="gross_energy_j") -> pd.DataFrame:
    rows = []
    for context, context_frame in frame.groupby("context_target"):
        pivot = context_frame.pivot_table(index="block", columns="config", values=metric, aggfunc="first").dropna()
        if pivot.shape[1] >= 3 and pivot.shape[0] >= 5:
            statistic, p_value = friedmanchisquare(*(pivot[column].to_numpy() for column in pivot.columns))
            kendall_w = statistic / (pivot.shape[0] * (pivot.shape[1] - 1))
            rows.append({
                "context": context, "metric": metric, "test": "Friedman",
                "n_blocks": pivot.shape[0], "n_configs": pivot.shape[1],
                "statistic": statistic, "p_value": p_value, "kendall_w": kendall_w,
            })
    return pd.DataFrame(rows)

performance_omnibus = performance_omnibus_tests(perf) if not perf.empty else pd.DataFrame()
performance_omnibus.to_csv(ROOT / "performance_omnibus_tests.csv", index=False)

def matched_binary_tests(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if frame.empty:
        return pd.DataFrame()
    frame = frame.copy()
    frame["correct"] = frame["correct"].astype(str).str.lower().isin(["true", "1"])
    for context, context_frame in frame.groupby("context_target"):
        pivot = context_frame.pivot_table(index="item_id", columns="config", values="correct", aggfunc="first").dropna()
        configs = list(pivot.columns)
        for i in range(len(configs)):
            for j in range(i + 1, len(configs)):
                first, second = configs[i], configs[j]
                table = pd.crosstab(pivot[first], pivot[second]).reindex(index=[False, True], columns=[False, True], fill_value=0)
                result = mcnemar(table.to_numpy(), exact=True)
                rows.append({
                    "context": context, "config_a": first, "config_b": second, "n_pairs": len(pivot),
                    "a_wrong_b_right": int(table.loc[False, True]), "a_right_b_wrong": int(table.loc[True, False]),
                    "statistic": result.statistic, "p_raw": result.pvalue,
                })
    output = pd.DataFrame(rows)
    if not output.empty:
        output["p_bonferroni"] = np.minimum(output["p_raw"] * len(output), 1.0)
    return output

niah_tests = matched_binary_tests(niah)
thinking_tests = matched_binary_tests(thinking)
niah_tests.to_csv(ROOT / "niah_mcnemar_tests.csv", index=False)
thinking_tests.to_csv(ROOT / "thinking_mcnemar_tests.csv", index=False)

display(perf_summary)
display(accuracy_summary)
display(performance_omnibus)
display(performance_tests)

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
if not perf_summary.empty:
    fig, axes = plt.subplots(2, 2, figsize=(18, 13))
    sns.lineplot(data=perf_summary, x="context", y="energy_mean_j", hue="config", marker="o", ax=axes[0, 0])
    axes[0, 0].set(title="Gross GPU Energy", ylabel="Energy (J)", xlabel="Input tokens")
    sns.lineplot(data=perf_summary, x="context", y="peak_vram_mean_mb", hue="config", marker="o", ax=axes[0, 1])
    axes[0, 1].set(title="Peak Allocated VRAM", ylabel="MiB", xlabel="Input tokens")
    sns.lineplot(data=perf_summary, x="context", y="decode_tok_s_mean", hue="config", marker="o", ax=axes[1, 0])
    axes[1, 0].set(title="Greedy Decode Throughput", ylabel="tokens/s", xlabel="Input tokens")
    sns.lineplot(data=perf_summary, x="context", y="cache_live_mean_mb", hue="config", marker="o", ax=axes[1, 1])
    axes[1, 1].set(title="Reachable Live Cache Storage", ylabel="MiB", xlabel="Input tokens")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "performance_overview.png", dpi=200, bbox_inches="tight")
    plt.show()

if not accuracy_summary.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=accuracy_summary, x="context", y="accuracy", hue="config")
    plt.ylim(0, 1.05)
    plt.title("Exact Accuracy by Context and Configuration")
    plt.ylabel("Accuracy")
    plt.xlabel("Input tokens")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "accuracy.png", dpi=200, bbox_inches="tight")
    plt.show()

# Pareto frontier at 8192: a point is dominated if another has no higher energy and no lower accuracy.
if not perf_summary.empty and not niah.empty:
    energy_8k = perf_summary[perf_summary["context"] == 8192][["config", "energy_mean_j"]]
    acc_8k = accuracy_summary[(accuracy_summary["experiment"] == "niah") & (accuracy_summary["context"] == 8192)][["config", "accuracy"]]
    pareto = energy_8k.merge(acc_8k, on="config")
    pareto["pareto"] = True
    for index, row in pareto.iterrows():
        dominated = ((pareto["energy_mean_j"] <= row["energy_mean_j"]) & (pareto["accuracy"] >= row["accuracy"]) & ((pareto["energy_mean_j"] < row["energy_mean_j"]) | (pareto["accuracy"] > row["accuracy"]))).any()
        pareto.loc[index, "pareto"] = not dominated
    pareto.to_csv(ROOT / "pareto_8192.csv", index=False)
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=pareto, x="energy_mean_j", y="accuracy", hue="config", style="pareto", s=180)
    for _, row in pareto.iterrows():
        plt.annotate(row["config"], (row["energy_mean_j"], row["accuracy"]), xytext=(6, 6), textcoords="offset points")
    plt.title("8192-token Energy-Accuracy Pareto Analysis")
    plt.xlabel("Mean gross GPU energy (J), lower is better")
    plt.ylabel("NIAH accuracy, higher is better")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "pareto_8192.png", dpi=200, bbox_inches="tight")
    plt.show()

## 14. Completeness and provenance report

In [ ]:
def completeness_report() -> pd.DataFrame:
    rows = []
    expected_perf = PERFORMANCE_BLOCKS
    expected_niah = NIAH_ITEMS_PER_CONDITION
    all_configs = ["C0", "C1", "C2", "C2_HQQ", "C3_snapkv", "C3", "C4"]
    for code in all_configs:
        cfg = CONFIGS.get(code)
        if cfg is None:
            continue
        for context in CONTEXT_LENGTHS:
            perf_n = len(perf[(perf["config"] == code) & (perf["context_target"] == context)]) if not perf.empty else 0
            niah_n = len(niah[(niah["config"] == code) & (niah["context_target"] == context)]) if not niah.empty else 0
            rows.append({
                "config": code, "context": context, "requested_method": cfg.requested_method,
                "actual_method": cfg.actual_method, "validated": cfg.method_validated,
                "performance_valid": perf_n, "performance_expected": expected_perf,
                "niah_valid": niah_n, "niah_expected": expected_niah,
                "blocked_reason": cfg.blocked_reason,
            })
    return pd.DataFrame(rows)

completeness = completeness_report()
completeness.to_csv(ROOT / "completeness.csv", index=False)
display(completeness)

provenance = {
    "generated_utc": utc_now(),
    "environment": MANIFEST,
    "idle_power_w": IDLE_POWER_W,
    "configs": {code: asdict(cfg) | {"cache_factory": getattr(cfg.cache_factory, "__name__", None) if cfg.cache_factory else "custom_cache"} for code, cfg in CONFIGS.items() if code in ["C0", "C1", "C2", "C2_HQQ", "C3_snapkv", "C3", "C4"]},
    "diagnostic_layerwise_budgets": CUSTOM_LAYERWISE_BUDGETS,
    "snapkv_config": {
        "window_size": SNAPKV_CONFIG.window_size,
        "kernel_size": SNAPKV_CONFIG.kernel_size,
        "max_capacity_prompt": SNAPKV_CONFIG.max_capacity_prompt,
    },
    "limitations": [
        "The custom position-aware selector is diagnostic only and is not reported as SnapKV or PyramidKV.",
        "Position-aware causal mask used for non-contiguous evicted caches.",
        "TurboQuant is a community implementation; HQQ is a distinct exploratory configuration and never substitutes for C2.",
        "GPU-only NVML energy excludes host CPU, cooling, networking, and embodied carbon.",
        "Ten performance blocks are below the course's aspirational n>=30 and limit statistical power.",
    ],
}
(ROOT / "experiment_provenance.json").write_text(json.dumps(provenance, indent=2, default=str), encoding="utf-8")
print(f"All artifacts saved under: {ROOT}")